# US Post-Market

## 1. Settings

In [8]:
import io
import logging
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone
from urllib.parse import quote_plus
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv
from IPython.display import Markdown, display

# 《股市》區塊
INDICES = {"^DJI": "道瓊", "^GSPC": "標普", "^IXIC": "納指", "^SOX": "費半", "^VIX": "VIX"}
ADRS = {"TSM": "台積", "UMC": "聯電", "ASX": "日月光"}           # 台股 ADR
# GICS 類股：維基百科名稱 → (中文, S&P 500 類股指數, 備援 SPDR ETF)
SECTORS = {
    "Information Technology": ("資訊科技", "^SP500-45", "XLK"),
    "Communication Services": ("通訊服務", "^SP500-50", "XLC"),
    "Consumer Discretionary": ("非必需消費", "^SP500-25", "XLY"),
    "Consumer Staples": ("必需消費", "^SP500-30", "XLP"),
    "Energy": ("能源", None, "XLE"),                 # Yahoo 沒有 S&P 500 能源類股指數，直接用 XLE
    "Financials": ("金融", "^SP500-40", "XLF"),
    "Health Care": ("醫療保健", "^SP500-35", "XLV"),
    "Industrials": ("工業", "^SP500-20", "XLI"),
    "Materials": ("原材料", "^SP500-15", "XLB"),
    "Utilities": ("公用事業", "^SP500-55", "XLU"),
    "Real Estate": ("不動產", "^SP500-60", "XLRE"),
}
FOCUS = {"GS": "Goldman Sachs", "MSFT": "Microsoft", "GOOGL": "Google", "MS": "Morgan Stanley", "AMZN": "Amazon",
         "JPM": "JP Morgan", "META": "Meta", "AAPL": "Apple", "TSLA": "Tesla", "AMD": "AMD", "NVDA": "NVIDIA",
         "SMCI": "SuperMicro"}
ASSETS = {"^TNX": "美債", "DX-Y.NYB": "美元", "GC=F": "黃金", "CL=F": "西德州", "BZ=F": "布蘭特"}
BREADTH = ["RSP", "SPY"]           # 等權重 vs 市值加權
logging.getLogger("yfinance").setLevel(logging.CRITICAL)   # 隱藏 yfinance 的逐檔錯誤訊息，改由第 2 格整理

MOVERS_PER_SECTOR = 3   # 強弱類股各列出幾檔領漲 / 領跌
NEWS_HOURS = 20         # 新聞抓最近幾小時
NEWS_PER_GROUP = 8
SP500_CACHE = "sp500_constituents.csv"
TIMEOUT = 15
TW = ZoneInfo("Asia/Taipei")
UA = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                     "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")}

# 金鑰放在同資料夾的 .env（不要寫進筆記本）
load_dotenv(".env")
FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY", "").strip()   # 個股新聞、財報、綜合新聞
FRED_API_KEY = os.getenv("FRED_API_KEY", "").strip()         # 美債殖利率備援
print("Finnhub：", "已設定" if FINNHUB_API_KEY else "未設定", "｜FRED：", "已設定" if FRED_API_KEY else "未設定")


def clean(text, limit=200):
    text = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", text or "")).strip()
    return text if len(text) <= limit else text[:limit].rstrip() + "…"


def safe_err(e):
    """錯誤訊息遮蔽金鑰"""
    msg = f"{type(e).__name__}: {e}"
    for k in (FINNHUB_API_KEY, FRED_API_KEY):
        if k:
            msg = msg.replace(k, "***")
    return clean(msg, 120)


def finnhub(path, **params):
    if not FINNHUB_API_KEY:
        raise ValueError("未設定 FINNHUB_API_KEY")
    r = requests.get(f"https://finnhub.io/api/v1/{path}", params={**params, "token": FINNHUB_API_KEY},
                     headers=UA, timeout=TIMEOUT)
    r.raise_for_status()
    data = r.json()
    if isinstance(data, dict) and data.get("error"):
        raise ValueError(data["error"])
    return data

Finnhub： 已設定 ｜FRED： 已設定


## 2. Prices

In [9]:
def closes(symbols, retry=True):
    symbols = list(symbols)
    df = yf.download(symbols, period="10d", interval="1d", auto_adjust=False, progress=False)
    if df.empty or "Close" not in df:
        raise RuntimeError("Yahoo Finance 沒有回傳資料（請檢查網路）")
    c = df["Close"]
    c = c.to_frame(symbols[0]) if isinstance(c, pd.Series) else c
    empty = [s for s in symbols if s not in c or c[s].dropna().empty]
    if retry and empty and len(empty) < len(symbols):     # 逾時等暫時性失敗：沒抓到的再試一次
        try:
            again = closes(empty, retry=False)
            c = c.drop(columns=[s for s in empty if s in c]).join(again, how="outer")
        except RuntimeError:
            pass
    return c


def last_two(c, as_of):
    """{代號: (收盤, 前一日收盤)}，以美股收盤日為準（期貨可能已有隔日資料）"""
    out = {}
    for s in c.columns:
        col = c[s].loc[:as_of].dropna()
        if len(col) >= 2:
            out[s] = (float(col.iloc[-1]), float(col.iloc[-2]))
    return out


def chg(s, q=None):
    last, prev = (q or Q)[s]
    return last - prev, (last / prev - 1) * 100, last


symbols = [*INDICES, *ADRS, *FOCUS, *ASSETS, *BREADTH] + [x for _, idx, etf in SECTORS.values() for x in (idx, etf) if x]
C = closes(symbols)
AS_OF = C[[s for s in INDICES if s in C]].dropna(how="all").index[-1]
Q = last_two(C, AS_OF)
# 類股漲跌：優先用 S&P 500 類股指數，沒有資料才用 SPDR ETF
SECTOR_PCT, via_etf, no_sector = {}, [], []
for en, (zh, idx, etf) in SECTORS.items():
    s = idx if idx in Q else etf if etf in Q else None
    if s:
        SECTOR_PCT[en] = chg(s)[1]
        if s == etf:
            via_etf.append(f"{zh}（{etf}）")
    else:
        no_sector.append(zh)
RANKED = sorted(SECTOR_PCT, key=SECTOR_PCT.get, reverse=True)
missing = [s for s in [*INDICES, *ADRS, *FOCUS, *ASSETS, *BREADTH] if s not in Q] + no_sector
print(f"資料日期：{AS_OF:%Y-%m-%d}（美東收盤）")
print("類股改用 ETF：" + "、".join(via_etf) if via_etf else "類股全部使用 S&P 500 類股指數")
print("缺資料：" + "、".join(missing) if missing else "全部抓取成功")

資料日期：2026-11-09（美東收盤）
類股改用 ETF：資訊科技（XLK）、通訊服務（XLC）、非必需消費（XLY）、必需消費（XLP）、能源（XLE）、金融（XLF）、醫療保健（XLV）、工業（XLI）、原材料（XLB）、公用事業（XLU）
全部抓取成功


## 3. Report Data Blocks

In [11]:
def updown(d, digits=2):
    return f"{'漲' if d >= 0 else '跌'}{abs(d):.{digits}f}"


out = ["《股市》", ""]
for s, name in {**INDICES, **ADRS}.items():
    if s not in Q:
        out += [f"🔘{name} n/a", ""]
        continue
    d, p, last = chg(s)
    if s == "^VIX":
        out += [f"🔘{name} {updown(d, 1)} ({p:+.2f}%) 收 {last:.2f}", ""]
    elif s in INDICES:
        out += [f"🔘{name} {updown(d)} ({p:+.2f}%) 收 {last:.0f}", ""]
    else:
        out += [f"🔘{name} {updown(d)} ({p:+.2f}%) 收 {last:.2f}", ""]
out += ["（盤勢評論：盤前 → 盤中 → 收盤，2–4 段）", "", "（類股評論：最強類股 → 次強 → 最弱，各一段）", ""]
out += ["🔹強勢類股：", ""] + [x for s in RANKED[:3] for x in (f"{SECTORS[s][0]} {SECTOR_PCT[s]:+.2f}%", "")]
out += ["🔹弱勢類股：", ""] + [x for s in RANKED[::-1][:3] for x in (f"{SECTORS[s][0]} {SECTOR_PCT[s]:+.2f}%", "")]
focus = sorted((s for s in FOCUS if s in Q), key=lambda s: chg(s)[1], reverse=True)
out += ["🔹焦點股： " + "、".join(f"{FOCUS[s]} {chg(s)[1]:+.2f}%" for s in focus), ""]
out += ["《美債/美元/黃金/原油》", ""]
for s, name in ASSETS.items():
    if s not in Q:
        out += [f"🔘{name} n/a", ""]
    elif s == "^TNX":
        last, prev = Q[s]
        out += [f"🔘{name} {round((last - prev) * 100):+d}bps 收{last:.2f}%", ""]
    else:
        d, p, last = chg(s)
        out += [f"🔘{name} {p:+.2f}% 收 {last:.2f}", ""]
out += [".", ""] + [f"🔹{k}：（待撰寫）\n" for k in ("美債", "美元", "黃金", "原油")]
out += ["《重點訊息》", "", "🔹利多：", "", "1. （待撰寫）", "", "🔹利空：", "", "1. （待撰寫）"]
print("\n".join(out))

《股市》

🔘道瓊 漲1104.02 (+2.15%) 收 52390

🔘標普 漲173.08 (+2.35%) 收 7550

🔘納指 漲673.71 (+2.74%) 收 25253

🔘費半 漲643.68 (+6.85%) 收 10034

🔘VIX 跌5.7 (-22.31%) 收 19.92

🔘台積 漲13.18 (+3.51%) 收 388.62

🔘聯電 漲1.04 (+7.51%) 收 14.89

🔘日月光 漲2.73 (+8.63%) 收 34.36

（盤勢評論：盤前 → 盤中 → 收盤，2–4 段）

（類股評論：最強類股 → 次強 → 最弱，各一段）

🔹強勢類股：

工業 +3.82%

資訊科技 +3.73%

非必需消費 +2.92%

🔹弱勢類股：

能源 +0.52%

必需消費 +0.90%

金融 +1.45%

🔹焦點股： AMD +13.26%、Tesla +5.54%、Meta +5.12%、Goldman Sachs +4.67%、Morgan Stanley +3.12%、Amazon +2.94%、Microsoft +2.75%、Apple +2.43%、JP Morgan +2.27%、Google +1.18%、SuperMicro -0.56%、NVIDIA -4.27%

《美債/美元/黃金/原油》

🔘美債 +0bps 收4.86%

🔘美元 -0.61% 收 101.68

🔘黃金 +1.24% 收 4125.60

🔘西德州 -5.43% 收 85.90

🔘布蘭特 -6.70% 收 91.73

.

🔹美債：（待撰寫）

🔹美元：（待撰寫）

🔹黃金：（待撰寫）

🔹原油：（待撰寫）

《重點訊息》

🔹利多：

1. （待撰寫）

🔹利空：

1. （待撰寫）


## 4. Sector detail, top movers, market breadth

In [12]:
def sp500():
    """S&P 500 成分股（維基百科），快取 7 天"""
    if os.path.exists(SP500_CACHE) and time.time() - os.path.getmtime(SP500_CACHE) < 7 * 86400:
        return pd.read_csv(SP500_CACHE)
    r = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers=UA, timeout=TIMEOUT)
    r.raise_for_status()
    t = next(t for t in pd.read_html(io.StringIO(r.text)) if {"Symbol", "GICS Sector"} <= set(t.columns))
    df = t.rename(columns={"Symbol": "symbol", "Security": "name", "GICS Sector": "sector",
                           "GICS Sub-Industry": "industry"})[["symbol", "name", "sector", "industry"]]
    df["symbol"] = df["symbol"].astype(str).str.replace(".", "-", regex=False)   # BRK.B → BRK-B
    df.to_csv(SP500_CACHE, index=False)
    return df


def tag(r):
    return f"{r['name']}（{r['symbol']} {r['chg']:+.2f}%）"


UNI = sp500()
QU = last_two(closes(UNI["symbol"]), AS_OF)
UNI["chg"] = [chg(s, QU)[1] if s in QU else None for s in UNI["symbol"]]
UNI = UNI.dropna(subset=["chg"])

adv, dec = int((UNI["chg"] > 0).sum()), int((UNI["chg"] < 0).sum())
lines = [f"**市場廣度**：上漲 {adv} 家、下跌 {dec} 家（漲跌家數比 {adv / max(dec, 1):.2f}）"]
if all(s in Q for s in BREADTH):
    lines[0] += f"｜等權重 RSP {chg('RSP')[1]:+.2f}% vs 市值加權 SPY {chg('SPY')[1]:+.2f}%"
MOVERS = []
for en in RANKED[:3] + RANKED[::-1][:3]:
    sub = UNI[UNI["sector"] == en].sort_values("chg", ascending=False)
    if sub.empty:
        continue
    ind = sub.groupby("industry")["chg"].agg(["mean", "count"]).sort_values("mean", ascending=False)
    fmt_ind = lambda rows: "、".join(f"{i} {m:+.2f}%（{int(n)}檔）" for i, (m, n) in rows.iterrows()) or "—"
    top = sub[sub["chg"] > 0].head(MOVERS_PER_SECTOR)
    bottom = sub[sub["chg"] < 0].tail(MOVERS_PER_SECTOR).iloc[::-1]
    for _, r in pd.concat([top, bottom]).iterrows():
        if r["symbol"] not in {m["symbol"] for m in MOVERS}:
            MOVERS.append(r)
    lines += ["", f"### {SECTORS[en][0]} {SECTOR_PCT[en]:+.2f}%",
              f"- 子產業最強：{fmt_ind(ind[ind['mean'] > 0].head(3))}",
              f"- 子產業最弱：{fmt_ind(ind[ind['mean'] < 0].tail(3).iloc[::-1])}",
              f"- 領漲：{'、'.join(tag(r) for _, r in top.iterrows()) or '—'}",
              f"- 領跌：{'、'.join(tag(r) for _, r in bottom.iterrows()) or '—'}"]
display(Markdown("\n".join(lines)))

**市場廣度**：上漲 423 家、下跌 78 家（漲跌家數比 5.42）｜等權重 RSP +2.27% vs 市值加權 SPY +2.37%

### 工業 +3.82%
- 子產業最強：Construction & Engineering +7.77%（4檔）、Trading Companies & Distributors +7.03%（3檔）、Heavy Electrical Equipment +6.95%（2檔）
- 子產業最弱：Research & Consulting Services -1.31%（2檔）、Human Resource & Employment Services -1.20%（2檔）
- 領漲：Bloom Energy（BE +11.91%）、Ferguson Enterprises（FERG +11.22%）、Vertiv（VRT +9.63%）
- 領跌：Axon Enterprise（AXON -3.14%）、Paychex（PAYX -1.96%）、Verisk Analytics（VRSK -1.95%）

### 資訊科技 +3.73%
- 子產業最強：Electronic Components +11.42%（3檔）、Semiconductor Materials & Equipment +10.15%（5檔）、Electronic Manufacturing Services +7.78%（3檔）
- 子產業最弱：Internet Services & Infrastructure -2.21%（3檔）、IT Consulting & Other Services -0.79%（4檔）
- 領漲：Coherent Corp.（COHR +16.79%）、Monolithic Power Systems（MPWR +16.60%）、Advanced Micro Devices（AMD +13.26%）
- 領跌：First Solar（FSLR -4.74%）、Nvidia（NVDA -4.27%）、Akamai Technologies（AKAM -3.37%）

### 非必需消費 +2.92%
- 子產業最強：Computer & Electronics Retail +6.67%（1檔）、Footwear +4.29%（1檔）、Automobile Manufacturers +3.41%（3檔）
- 子產業最弱：—
- 領漲：Best Buy（BBY +6.67%）、Ralph Lauren Corporation（RL +5.83%）、Tesla, Inc.（TSLA +5.54%）
- 領跌：Las Vegas Sands（LVS -1.35%）、eBay Inc.（EBAY -0.81%）、Yum! Brands（YUM -0.16%）

### 能源 +0.52%
- 子產業最強：Oil & Gas Refining & Marketing +3.51%（3檔）、Oil & Gas Storage & Transportation +2.52%（4檔）、Oil & Gas Equipment & Services +0.75%（3檔）
- 子產業最弱：Integrated Oil & Gas -0.77%（2檔）
- 領漲：Marathon Petroleum（MPC +4.60%）、Williams Companies（WMB +3.70%）、Targa Resources（TRGP +3.42%）
- 領跌：ConocoPhillips（COP -1.32%）、Occidental Petroleum（OXY -1.10%）、Chevron Corporation（CVX -0.86%）

### 必需消費 +0.90%
- 子產業最強：Household Products +3.01%（4檔）、Consumer Staples Merchandise Retail +2.57%（5檔）、Personal Care Products +2.29%（2檔）
- 子產業最弱：Tobacco -1.41%（2檔）、Packaged Foods & Meats -0.22%（8檔）
- 領漲：Clorox（CLX +5.57%）、Target Corporation（TGT +4.73%）、Dollar Tree（DLTR +4.64%）
- 領跌：Hershey Company (The)（HSY -2.58%）、Philip Morris International（PM -2.41%）、General Mills（GIS -1.77%）

### 金融 +1.45%
- 子產業最強：Asset Management & Custody Banks +3.71%（12檔）、Investment Banking & Brokerage +2.92%（6檔）、Life & Health Insurance +2.75%（5檔）
- 子產業最弱：Insurance Brokers -0.56%（6檔）、Consumer Finance -0.43%（3檔）
- 領漲：Invesco（IVZ +6.76%）、Franklin Resources（BEN +6.34%）、Apollo Global Management（APO +4.67%）
- 領跌：Capital One（COF -1.63%）、Brown & Brown（BRO -1.55%）、Fidelity National Information Services（FIS -1.36%）

## 5. Why stocks moved: analyst ratings + headlines

In [13]:
ACTION = {"up": "升評", "down": "降評", "init": "首評", "main": "維持", "reit": "重申"}
DAY = AS_OF.date()


def analyst(sym, days=3):
    """Yahoo 分析師評等變動（近 N 天）"""
    ud = yf.Ticker(sym).upgrades_downgrades
    if ud is None or ud.empty:
        return []
    idx = pd.to_datetime(ud.index)
    idx = idx.tz_localize(None) if idx.tz is not None else idx
    ud = ud[idx >= pd.Timestamp(AS_OF) - pd.Timedelta(days=days)]
    out = []
    for _, r in ud.iterrows():
        text = f"{r.get('Firm', '')} {ACTION.get(r.get('Action'), r.get('Action', ''))}"
        if r.get("FromGrade") and r.get("FromGrade") != r.get("ToGrade"):
            text += f"：{r['FromGrade']} → {r.get('ToGrade', '')}"
        elif r.get("ToGrade"):
            text += f"：{r['ToGrade']}"
        if pd.notna(r.get("currentPriceTarget")) and r.get("currentPriceTarget"):
            prior = r.get("priorPriceTarget")
            text += f"，目標價 {prior:g} → {r['currentPriceTarget']:g}" if pd.notna(prior) and prior else \
                f"，目標價 {r['currentPriceTarget']:g}"
        out.append(text)
    return out


def stock_news(sym, n=3):
    """有 Finnhub 金鑰用 Finnhub（含摘要），否則用 Yahoo"""
    if FINNHUB_API_KEY:
        data = finnhub("company-news", symbol=sym, **{"from": f"{DAY - timedelta(days=1):%Y-%m-%d}",
                                                       "to": f"{DAY + timedelta(days=1):%Y-%m-%d}"})
        data = sorted((d for d in data if d.get("headline")), key=lambda d: d.get("datetime", 0), reverse=True)[:n]
        return [f"[{clean(d['headline'], 200)}]({d.get('url', '')}) — {d.get('source', '')}"
                + (f"：{clean(d['summary'], 160)}" if d.get("summary") else "") for d in data]
    out = []
    for it in (yf.Ticker(sym).news or [])[:n]:
        c = it.get("content", it)            # yfinance 新舊兩種格式
        url = (c.get("canonicalUrl") or {}).get("url") or c.get("link", "")
        src = (c.get("provider") or {}).get("displayName") or c.get("publisher", "")
        if c.get("title"):
            out.append(f"[{c['title']}]({url}) — {src}")
    return out


def gather(r):
    info = {"r": r}
    for key, fn in (("analyst", analyst), ("news", stock_news)):   # 分開抓：一邊失敗不影響另一邊
        try:
            info[key] = fn(r["symbol"])
        except Exception as e:
            info[key + "_err"] = safe_err(e)
    return info


with ThreadPoolExecutor(max_workers=2) as ex:     # 同時太多請求會被 Yahoo 擋
    infos = list(ex.map(gather, MOVERS))

lines, analyst_errs = [], []
for x in infos:
    lines.append(f"**{tag(x['r'])}**")
    lines += [f"- 📊 {a}" for a in x.get("analyst", [])]
    lines += [f"- 📰 {n}" for n in x.get("news", [])]
    if "news_err" in x:
        lines.append(f"- 📰 （新聞無法取得：{x['news_err']}）")
    if "analyst_err" in x:
        analyst_errs.append(x["analyst_err"])
    if not x.get("analyst") and not x.get("news") and "news_err" not in x:
        lines.append("- （近期無評等變動或新聞）")
    lines.append("")
if analyst_errs:
    lines.append(f"⚠️ 分析師評等 {len(analyst_errs)}/{len(infos)} 檔無法取得（Yahoo 限流時常見，稍後重跑本格即可）：{analyst_errs[0]}")
display(Markdown("\n".join(lines)))

# ---- 當日財報（Finnhub）：寫「重點訊息」與「盤後」段落用 ----
HOUR = {"bmo": "盤前", "amc": "盤後", "dmh": "盤中"}


def beat(actual, est):
    if actual is None or est is None:
        return ""
    return "（優於預期）" if actual > est else "（不如預期）" if actual < est else "（符合預期）"


try:
    cal = finnhub("calendar/earnings", **{"from": f"{DAY:%Y-%m-%d}", "to": f"{DAY:%Y-%m-%d}"})["earningsCalendar"]
    sp = UNI.set_index("symbol")
    cal = sorted((d for d in cal if d.get("symbol") in sp.index), key=lambda d: -(d.get("revenueEstimate") or 0))
    rows = ["| 公司 | 時段 | EPS 實際 / 預估 | 營收 實際 / 預估（億美元） | 當日股價 |", "|:---|:---|---:|---:|---:|"]
    for d in cal[:20]:
        rev = lambda v: f"{v / 1e8:,.1f}" if v else "—"
        eps = lambda v: f"{v:.2f}" if v is not None else "—"
        px = f"{sp.loc[d['symbol'], 'chg']:+.2f}%" if pd.notna(sp.loc[d["symbol"], "chg"]) else "—"
        rows.append(f"| {sp.loc[d['symbol'], 'name']}（{d['symbol']}） | {HOUR.get(d.get('hour'), d.get('hour') or '—')} | "
                    f"{eps(d.get('epsActual'))} / {eps(d.get('epsEstimate'))}{beat(d.get('epsActual'), d.get('epsEstimate'))} | "
                    f"{rev(d.get('revenueActual'))} / {rev(d.get('revenueEstimate'))} | {px} |")
    body = "\n".join(rows) if cal else "- 當日無 S&P 500 成分股公布財報"
    note = "\n\n盤後公布的財報，當日股價尚未反映，實際數字也可能稍晚才更新。" if cal else ""
except Exception as e:
    body, note = f"- 無法取得：{safe_err(e)}", ""
display(Markdown(f"**當日財報｜{DAY:%Y-%m-%d}（S&P 500 成分股，依營收規模排序）**\n\n{body}{note}"))

**Bloom Energy（BE +11.91%）**
- 📊 RBC Capital 重申：Outperform，目標價 335 → 335
- 📰 [Stock Market Today, Nov. 9: Plug Power Rises 3.45% on Positioning Ahead of Q3 Report](https://finnhub.io/api/news?id=dd7f22519ccfdee024f2ce0a4d9666e5d02a1ed5bb78277b95639b080d8764fa) — Yahoo：Today, Nov. 9, 2026, traders weighed Plug Power's upcoming results against a backdrop of tight margins and steady cash use.
- 📰 [Is Bloom Energy Really Worth $200 Right Now? Here's the Math.](https://finnhub.io/api/news?id=d53556811262f7f87fb27f24f6ecfc4c21fbcf44230e7002900922219647213a) — Yahoo：Bloom Energy has delivered explosive growth as demand for its power solutions surges. But even after its stock has fallen sharply, investors need to consider ho…
- 📰 [S&P 500 Top Movers: Gainers and Losers for Nov. 9, 2026](https://finnhub.io/api/news?id=8dbe13d7bad59b65126f092b011cd055233dfd03d70a27cbf9c95faf7c97444b) — ChartMill：S&P 500 top movers for Nov. 9, 2026 include COHR's 17.41% gain, NVDA's 7.27% decline, and moves in MPWR, AMD, FSLR and AXON.

**Ferguson Enterprises（FERG +11.22%）**
- 📰 [Ferguson plc Q3 2026 Earnings Call Summary](https://finnhub.io/api/news?id=4575aab8b3814fe84be851d4f03b32019ec1cd6f71a31f8a21f500886145930c) — Yahoo：Moby summary of Ferguson plc's Q3 2026 earnings call
- 📰 [Ferguson (FERG) Q3 2026 Earnings Call Transcript](https://finnhub.io/api/news?id=81a178b8db2c42064167bc6b8e9b9120d693d50a796cf331e73bdb3265f021d4) — Yahoo：Nonresidential strength offset housing softness, and full-year sales guidance rose.
- 📰 [Ferguson Enterprises Inc (FERG) (Q3 2026) Earnings Call Highlights: Sales Up 9. ...](https://finnhub.io/api/news?id=8fd83bc6ef63ecc1c41c47eb941f978d3dca6507ce7e8918d6bd84d6ef426c72) — Yahoo：Ferguson Enterprises Inc (FERG) posted 7% organic growth with industrial sales up 55% and nonresidential up 15%, prompting management to lift full-year net sale…

**Vertiv（VRT +9.63%）**
- 📰 [Eaton Lost 23% in a Month. RBC Sees 58% Upside From Here](https://finnhub.io/api/news?id=bdddc912998a04a6557924c7a06bc9d8fd4089271a591b332fdcfbd679b276c3) — Yahoo：Key TakeawaysEaton stock is down 23% in the past month. Meta Platforms’ spending plans and the Democrats’ midterm sweep hit AI power-equipment stocks across the…
- 📰 [Seagate Soars 10% in AI Rebound as Morgan Stanley Sees “Strong Fundamentals and Weak Sentiment”](https://finnhub.io/api/news?id=3aa955dd8c45f0c1728b062676c023ff1fae133d87f5d9ea4d49352db5365210) — Yahoo：Key TakeawaysSeagate shares jumped 10% today as AI hardware stocks, including Coherent, bounced back from last week’s selloff. Morgan Stanley kept Seagate as a…
- 📰 [GE Vernova Soars 7% as Constellation Raises Guidance, Easing Data-Center Moratorium Fears](https://finnhub.io/api/news?id=5998360c3c0a5da0fcfc8a66b5cb538cac2425f2fa8377320e1aae51ad238198) — Yahoo：Key TakeawaysGE Vernova shares jumped 7% today, with Bloom Energy and Vertiv, after Constellation Energy beat estimates and raised its 2026 guidance. Constellat…

**Axon Enterprise（AXON -3.14%）**
- 📊 Barclays 維持：Overweight，目標價 688 → 507
- 📊 JP Morgan 維持：Overweight，目標價 755 → 610
- 📊 RBC Capital 維持：Outperform，目標價 735 → 615
- 📊 Piper Sandler 重申：Overweight，目標價 561 → 540
- 📊 Needham 維持：Buy，目標價 750 → 500
- 📰 [Axon Enterprise: 3 Key Takeaways for Long-Term Investors](https://finnhub.io/api/news?id=0144741054d719fcbe44cdc2082dfc81bf74b7d8e9eca3a2015206f56d10c42e) — Yahoo：Axon's $15.8 billion in future contracted bookings gives the company a multi-year runway, but its inventory buildup is already weighing on free cash flow.
- 📰 [S&P 500 Top Movers: Gainers and Losers for Nov. 9, 2026](https://finnhub.io/api/news?id=8dbe13d7bad59b65126f092b011cd055233dfd03d70a27cbf9c95faf7c97444b) — ChartMill：S&P 500 top movers for Nov. 9, 2026 include COHR's 17.41% gain, NVDA's 7.27% decline, and moves in MPWR, AMD, FSLR and AXON.
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=20d8a95fb6d88aa6d92d94f3823a12edc8192a1213a79a8e22a78db334a9693f) — ChartMill：S&P 500 top gainers and losers from today's US session, with percentage moves and latest prices for each listed ticker.

**Paychex（PAYX -1.96%）**
- 📰 [S&P 500 Opening Gap Movers for Nov. 9, 2026](https://finnhub.io/api/news?id=d9200800176708874dd109c854e6615a2ba6f51cf2b81ae83fce100f0882c5b8) — ChartMill：S&P 500 opening gap movers for Nov. 9, 2026, including gap-up names such as FERG, AMD and NRG, and gap-down names including NVDA, HSY and FSLR.
- 📰 [Paychex Never Cut Its Dividend During the 2008 Crisis. Today It Pays Nearly 4 Times as Much](https://finnhub.io/api/news?id=91422935282d219162b1ab2786b431796a3d51a6bcba86afa5cf4e27fe603ec8) — Yahoo：Paychex held its payout steady through the worst recession in decades, but this time it carries billions in new debt and a dividend that already outruns quarter…
- 📰 [Social Security Reform Could Put ADP Stock Back On More Watchlists](https://finnhub.io/api/news?id=462701dbc69e1f0c5bf470d2b88924a6abc20b91876cf1676c64e44fae094775) — Yahoo：Social Security reform has suddenly shifted from a distant talking point to a live policy question, and payroll and HR services stocks are right in that crossfi…

**Verisk Analytics（VRSK -1.95%）**
- 📊 Barclays 維持：Overweight，目標價 225 → 250
- 📊 Wells Fargo 維持：Overweight，目標價 260 → 265
- 📰 [QXO is Set to Report Q3 Earnings: Here's What Investors Should Know](https://finnhub.io/api/news?id=bdf3151d4d778931f69f127be5dba89cbbc93e8340a48d627567d2f7f8c435e1) — Yahoo：QXO is set to report Q3 earnings on Nov. 12, with revenues poised to surge 77.6% on acquisitions, while higher costs and interest expenses may weigh on profits.

**Coherent Corp.（COHR +16.79%）**
- 📊 Citigroup 維持：Buy，目標價 420 → 440
- 📰 [Stock Market Today: Dow Rallies As Oil Prices Slide; Nvidia Stock Dives On CEO Huang Probe (Live Coverage)](https://finnhub.io/api/news?id=9ed519a9581c0b645208639091ac07e8066a724f30fe3e5d74827003446af2c6) — Yahoo：The Dow Jones index rallies 500 points Monday as oil prices drop. Nvidia stock dives on the probe into CEO Jensen Huang.
- 📰 [Nvidia, AMD, CoreWeave, Ionis, DraftKings, and More Stocks That Explain Today’s Market](https://finnhub.io/api/news?id=df4ae887cc7b016b4b827242c67462e0ce46d0f2138d0b119ab52fb1077de05f) — Yahoo：Nvidia falls after the chip maker’s board launches an investigation into reports that CEO Jensen Huang was targeted by a suspected Chinese spy.
- 📰 [US Equity Markets End Higher as Oil Drops; Industrials, Consumer Discretionary Lead Gains](https://finnhub.io/api/news?id=076cc2f72d73ef61a44dc704d55c57fdf1d7dcd9ba1399a054d1c519a006a85a) — Yahoo：US equity indexes ended higher Monday as crude oil prices fell, with industrials and consumer discre

**Monolithic Power Systems（MPWR +16.60%）**
- 📰 [Stock Market Today: Dow Rallies As Oil Prices Slide; Nvidia Stock Dives On CEO Huang Probe (Live Coverage)](https://finnhub.io/api/news?id=9ed519a9581c0b645208639091ac07e8066a724f30fe3e5d74827003446af2c6) — Yahoo：The Dow Jones index rallies 500 points Monday as oil prices drop. Nvidia stock dives on the probe into CEO Jensen Huang.
- 📰 [S&P 500 Top Movers: Gainers and Losers for Nov. 9, 2026](https://finnhub.io/api/news?id=8dbe13d7bad59b65126f092b011cd055233dfd03d70a27cbf9c95faf7c97444b) — ChartMill：S&P 500 top movers for Nov. 9, 2026 include COHR's 17.41% gain, NVDA's 7.27% decline, and moves in MPWR, AMD, FSLR and AXON.
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=20d8a95fb6d88aa6d92d94f3823a12edc8192a1213a79a8e22a78db334a9693f) — ChartMill：S&P 500 top gainers and losers from today's US session, with percentage moves and latest prices for each listed ticker.

**Advanced Micro Devices（AMD +13.26%）**
- 📊 Oppenheimer 升評：Perform → Outperform，目標價 600
- 📰 [The AI Trade Looks Like It's Falling Apart. AMD's Earnings Delivered a Reality Check.](https://finnhub.io/api/news?id=07fdb88990dd582c711543dad5ad060ce669a3c3240c5d7e783d370648ac943a) — Yahoo：AMD is still growing and is optimistic about the fourth quarter, despite recent market noise about the AI trade.
- 📰 [Stock Market Today, Nov. 9: AMD Jumps on Oppenheimer Upgrade and Google Cloud Chip Selection](https://finnhub.io/api/news?id=08b8056d2964ad1cea8a13e77e89d7e6982e6dcb891ac17068100f42d5982901) — Yahoo：Today, Nov. 9, 2026, shares of AI-focused chip designers faced a crucial week as investors weigh data-center demand ahead of next week's sector earnings.
- 📰 [AI infrastructure spending is still accelerating—but can it last?](https://finnhub.io/api/news?id=a0d61355196265f539e7dd4b5917ebcf0a361633c15ba2df12ec8b6015b49da0) — Yahoo：Bernstein Managing Director and Senior Analyst Stacy Rasgon joins Deirdre Bosa Live to break down the biggest risks facing the AI trade, what investors should b…

**First Solar（FSLR -4.74%）**
- 📊 Piper Sandler 維持：Overweight，目標價 251 → 222
- 📰 [Nvidia, AMD, CoreWeave, Ionis, DraftKings, and More Stocks That Explain Today’s Market](https://finnhub.io/api/news?id=df4ae887cc7b016b4b827242c67462e0ce46d0f2138d0b119ab52fb1077de05f) — Yahoo：Nvidia falls after the chip maker’s board launches an investigation into reports that CEO Jensen Huang was targeted by a suspected Chinese spy.
- 📰 [S&P 500 Top Movers: Gainers and Losers for Nov. 9, 2026](https://finnhub.io/api/news?id=8dbe13d7bad59b65126f092b011cd055233dfd03d70a27cbf9c95faf7c97444b) — ChartMill：S&P 500 top movers for Nov. 9, 2026 include COHR's 17.41% gain, NVDA's 7.27% decline, and moves in MPWR, AMD, FSLR and AXON.
- 📰 [First Solar Stock Is the Worst in the S&P 500 Today. Was Blue Wave Optimism Premature?](https://finnhub.io/api/news?id=38b30f4953ee1cd03e6f2c72189bd40d393794df2bc61193fc02a3309232b841) — Yahoo：First Solar stock soared after Democratic Party victories last week, but analysts suggest a rally might have come too early.

**Nvidia（NVDA -4.27%）**
- 📊 RBC Capital 重申：Outperform，目標價 330 → 330
- 📊 Goldman Sachs 重申：Buy，目標價 300 → 300
- 📰 [Is Wall Street Turning on AI?](https://finnhub.io/api/news?id=823ba12888a38b1e7a551dbfc81ac687d11ea4e328049a7144f5e5bb3b1d80d2) — Yahoo：Alphabet’s sharp sell-off raises a big question: Are investors more concerned about AI spending than AI growth?
- 📰 [We Are Witnessing the Stock Market Do Something for Only the 2nd Time in 136 Years, and History Is Clear What Comes Next for Stocks](https://finnhub.io/api/news?id=e17942e210b3e0a0539323136a367844b4c8ae35bb03a7c02fc30b5abca383a1) — Yahoo：After over four years of strong returns, the stock market looks pricey based on some closely watched metrics.
- 📰 [Why Lincoln Educational Stock Surged Today](https://finnhub.io/api/news?id=df1ffa3770f96a6f013360dec55b81ee2b83b178defc1eb43db2dc3c57187afe) — Yahoo：Students are demanding a higher return on their investments.

**Akamai Technologies（AKAM -3.37%）**
- 📊 Baird 維持：Neutral，目標價 140 → 92
- 📊 UBS 維持：Neutral，目標價 100 → 90
- 📊 DA Davidson 維持：Buy，目標價 150 → 125
- 📊 Scotiabank 維持：Sector Outperform，目標價 180 → 120
- 📊 RBC Capital 維持：Sector Perform，目標價 135 → 85
- 📊 Evercore ISI Group 維持：Outperform，目標價 175 → 150
- 📊 Keybanc 維持：Overweight，目標價 195 → 175
- 📊 Susquehanna 維持：Positive，目標價 140 → 90
- 📰 [S&P 500 Top Movers: Gainers and Losers for Nov. 9, 2026](https://finnhub.io/api/news?id=8dbe13d7bad59b65126f092b011cd055233dfd03d70a27cbf9c95faf7c97444b) — ChartMill：S&P 500 top movers for Nov. 9, 2026 include COHR's 17.41% gain, NVDA's 7.27% decline, and moves in MPWR, AMD, FSLR and AXON.
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=20d8a95fb6d88aa6d92d94f3823a12edc8192a1213a79a8e22a78db334a9693f) — ChartMill：S&P 500 top gainers and losers from today's US session, with percentage moves and latest prices for each listed ticker.
- 📰 [Moody’s and S&P downgrade Akamai on debt-fueled AI infrastructure buildout](https://finnhub.io/api/news?id=42334aaf5df06138a98b9f2a1a42ce184fa15c8163c2b5492c637ff2e75f263b) — Yahoo：Investing.com -- Akamai Technologies Inc (NASDAQ:AKAM) saw its credit profile take a significant hit on Monday as both Moody’s Ratings and S&P Global Ratings do…

**Best Buy（BBY +6.67%）**
- 📊 Citigroup 維持：Neutral，目標價 88 → 102
- 📊 Wells Fargo 維持：Equal-Weight，目標價 85 → 95
- 📰 [Why Best Buy (BBY) is a Top Dividend Stock for Your Portfolio](https://finnhub.io/api/news?id=978794430a564fe4820eec9ecf687ce5088d4adb4d82ccbc96c99c987c9f2b87) — Yahoo：Dividends are one of the best benefits to being a shareholder, but finding a great dividend stock is no easy task. Does Best Buy (BBY) have what it takes? Let's…
- 📰 [Should Value Investors Buy Best Buy (BBY) Stock?](https://finnhub.io/api/news?id=863fa46bb29f085e033f6b2f69886052a2c4947ab33a1653ce36fdad3066ad90) — Yahoo：Here at Zacks, our focus is on the proven Zacks Rank system, which emphasizes earnings estimates and estimate revisions to find great stocks. Nevertheless, we a…
- 📰 [Is Best Buy (BBY) Outperforming Other Retail-Wholesale Stocks This Year?](https://finnhub.io/api/news?id=c4eaf38059a650ad8f3331b2a3e0d32b69e80a31dba9ec25be96acdfd2169662) — Yahoo：Here is how Best Buy (BBY) and Casey's General Stores (CASY) have performed compared to their sector so far this year.

**Ralph Lauren Corporation（RL +5.83%）**
- 📊 Barclays 維持：Overweight，目標價 463 → 473
- 📰 [Ralph Lauren Warns Inflation-Weary European Shoppers Are Pulling Back](https://finnhub.io/api/news?id=4835a1a58235e24fd4890d92e3573ae4d2c30bcab12e5cd236c1465d2a4ce27e) — Yahoo：Ralph Lauren Corp. stock traded lower Thursday despite reporting better-than-expected second-quarter fiscal 2027 results and raising its full-year outlook. Howe…

**Tesla, Inc.（TSLA +5.54%）**
- 📰 [How much of Elon Musk’s wealth is in Tesla & SpaceX stock?](https://finnhub.io/api/news?id=0f6f51314e4f598ba8fffa0d83b4c2ff8bf0da9da76cdba042a421b245c889ea) — Yahoo：Elon Musk’s stakes in Tesla and SpaceX make him one of the world’s richest people. Here’s how much he owns in each company and what both stakes are worth togeth…
- 📰 [Securitize to Acquire River One, Expanding Into $6 Trillion Asset-Backed Finance Market](https://finnhub.io/api/news?id=4e681435fe0d52a66f763a47752107cbc2b79148f998033c2a41b2a3bfaf5c58) — Yahoo：Tokenization platform Securitize (NYSE: $SECZ) agreed Monday to acquire River One, an institutional asset-backed fi...
- 📰 [Dow Jones Soars 1,132 Points As Amazon, Tesla Rally, But Nvidia Dives; CPI Inflation Report Next](https://finnhub.io/api/news?id=d2812e27adcfde2a66b4303816dad8cacbaffcbaf4ea051663063ac89dc1e645) — Yahoo：Dow Jones futures were in focus after soaring 1,132 points ahead of the CPI inflation report. Amazon and Tesla rallied, but Nvidia dived.

**Las Vegas Sands（LVS -1.35%）**
- （近期無評等變動或新聞）

**eBay Inc.（EBAY -0.81%）**
- 📰 [Buy 5 Non-Tech Nasdaq Stocks to Enhance Your Portfolio Returns in Q4](https://finnhub.io/api/news?id=316a88d5e3d739892219f8bff5b80f34306bce3f4a0befa640592b0ebcdbebb8) — Yahoo：APA, CASY, EBAY, HAS and EQIX stand out for the rest of 2026, backed by improving earnings estimates and growth opportunities.

**Yum! Brands（YUM -0.16%）**
- 📰 [MCD Stock Outlook: Wall Street Eyes a Potential Upside Breakout After Q3 Results](https://finnhub.io/api/news?id=a69edb33d2d2ed79c6dc000d57ab4cbe4afdf2d8933e984e1ae75e395662aca4) — Yahoo：McDonald's stock bounced hard after earnings, yet U.S. guest counts are still sliding and revenue missed again. One variable could decide whether the rally runs…
- 📰 [AI is changing the KFC and Taco Bell experience for customers](https://finnhub.io/api/news?id=0f321bec1532f1fccab7cd528ff458076a87dbe677110b59e82d2d52892d33f1) — Yahoo：From who takes your order to how fast your food arrives, AI is reshaping two of America’s biggest fast-food chains.
- 📰 [McDonald’s Just Became a Dividend King. Is It Still a Buy for Retirement?](https://finnhub.io/api/news?id=e29f95904ef600207e861f089e851292c4d0e449228060f5ccb54f3c9c13e205) — Yahoo：Fifty years of dividend hikes sound bulletproof, but McDonald's U.S. traffic is sliding while debt costs climb. The cash coverage numbers and one balance sheet…

**Marathon Petroleum（MPC +4.60%）**
- 📰 [Seven OPEC+ Producers Keep December Output Unchanged While Crude Opens Lower](https://finnhub.io/api/news?id=c34103684845d936d59a3f01b8a42a45c00384cdaba695ab2290cd7cd5d754c2) — Yahoo：Crude oil futures were lower on Monday morning. As of 9:47 a.m. ET, the December NYMEX West Texas Intermediate contract was at $87.43 a barrel, down $3.40 or 3.…
- 📰 [CNQ Q3 Earnings Beat on Strong Pricing and Record Thermal Output](https://finnhub.io/api/news?id=b0f1b7cffc964722cdb418c8957ef61f0724fabda02b99f09c74d3b66172e18d) — Yahoo：Canadian Natural beats Q3 earnings estimates on stronger oil pricing and record thermal production, while surging free cash flow boosts shareholder returns.
- 📰 [Can Cenovus Improve Refining Margins Through Market Expansion?](https://finnhub.io/api/news?id=dc414eea498d374fb51e10679637a5547a74019975cd07df936415061a8c31be) — Yahoo：CVE is expanding product-market access and optimizing refinery output to capture stronger pricing opportunities and improve downstream margins.

**Williams Companies（WMB +3.70%）**
- 📰 [Two 13% Yields That Just Went On Clearance](https://finnhub.io/api/news?id=c38d36ccaf790d477bd0c3882b277e2bfc97ed859a8f82cf907d78637135cda2) — SeekingAlpha：Recent volatility has hit BDCs and MLPs hard, creating rare value in two 13% yielders.
- 📰 [Top Wall Street analysts like these 3 dividend stocks for consistent income](https://finnhub.io/api/news?id=cb272b66616961d114cfae5687ebb8a710c2cd9138da9e9974dbc5a0c590d836) — CNBC：Three dividend-paying stocks that are highlighted by Wall Street’s top pros, as tracked by TipRanks.

**Targa Resources（TRGP +3.42%）**
- 📊 Wells Fargo 維持：Overweight，目標價 324 → 330
- 📊 Jefferies 維持：Buy，目標價 326 → 311
- 📊 Citigroup 維持：Buy，目標價 325 → 338
- 📰 [Wells Fargo Maintains Overweight on Targa Resources, Raises Price Target to $330](https://finnhub.io/api/news?id=1ffd803264cc452c304c65b3847ea77d58480743d4b5ba1ccca9c4e85f11c4df) — Benzinga：Wells Fargo analyst Praneeth Satish maintains Targa Resources (NYSE:TRGP) with a Overweight and raises the price target from $324 to $330.
- 📰 [What Targa Resources (TRGP) Said on Its Q3 Earnings Call](https://finnhub.io/api/news?id=f251ec4c8f20224f4ee37d2ef109bd247d8dcfa878c833f3809e83dfaa28457d) — Yahoo：Targa Resources (NYSE:TRGP) expects 2026 adjusted earnings before interest, taxes, depreciation and amortization to reach at least $6 billion, as record Permian…

**ConocoPhillips（COP -1.32%）**
- 📊 Truist Securities 維持：Hold，目標價 136 → 142
- 📊 Wells Fargo 維持：Overweight，目標價 189 → 185
- 📰 [ConocoPhillips Trades at a Premium: Is It the Right Time to Buy?](https://finnhub.io/api/news?id=2385160493d8e4f7aa2160ec018894c6f23a0f6ed04102a54e779dd0dd2cc8fe) — Yahoo：COP's 54.4% stock rally, low-cost assets, LNG expansion and Willow project fuel optimism, but its premium valuation raises concerns for investors.
- 📰 [Seven OPEC+ Producers Keep December Output Unchanged While Crude Opens Lower](https://finnhub.io/api/news?id=c34103684845d936d59a3f01b8a42a45c00384cdaba695ab2290cd7cd5d754c2) — Yahoo：Crude oil futures were lower on Monday morning. As of 9:47 a.m. ET, the December NYMEX West Texas Intermediate contract was at $87.43 a barrel, down $3.40 or 3.…
- 📰 [COP vs. WHD: Which Stock Is the Better Value Option?](https://finnhub.io/api/news?id=c00405272c19f4430862a2b6d23182b8331de091fbeeb446b461395bda019ca1) — Yahoo：COP vs. WHD: Which Stock Is the Better Value Option?

**Occidental Petroleum（OXY -1.10%）**
- 📰 [Occidental (OXY) Reports Q3 Earnings: What Key Metrics Have to Say](https://finnhub.io/api/news?id=f4331ff0cc6f04468df7b330f51d6e44b1e81a6570486050a03426ec084fda81) — Yahoo：The headline numbers for Occidental (OXY) give insight into how the company performed in the quarter ended September 2026, but it may be worthwhile to compare s…
- 📰 [Dow Jones Soars 1,132 Points As Amazon, Tesla Rally, But Nvidia Dives; CPI Inflation Report Next](https://finnhub.io/api/news?id=d2812e27adcfde2a66b4303816dad8cacbaffcbaf4ea051663063ac89dc1e645) — Yahoo：Dow Jones futures were in focus after soaring 1,132 points ahead of the CPI inflation report. Amazon and Tesla rallied, but Nvidia dived.
- 📰 [Occidental beats Q3 forecasts as production offsets lower crude prices](https://finnhub.io/api/news?id=1018fabfaadb2ce317c6e1f6e16a27d1446f38285160c1fb5581f30236d266eb) — Yahoo：Investing.com -- Occidental Petroleum Corporation (NYSE:OXY) delivered a decisive third-quarter earnings beat on Monday, lifted by operational outperformance in…

**Chevron Corporation（CVX -0.86%）**
- 📊 Freedom Broker 升評：Sell → Hold，目標價 218
- 📰 [Exxon and Chevron Are Near Record Highs. Here's What History Says Happens Next.](https://finnhub.io/api/news?id=05ae32061fb485ae71c872fa0495e5607e9dedaa93b7d6e6b9234d0a5efcd80e) — Yahoo：These integrated energy giants are built to survive the entire energy cycle.
- 📰 [QatarEnergy Farm-In to PEL 82](https://finnhub.io/api/news?id=6d1b5bd8812082fdb6ed1014001e13f5b0a324d9c5458a17080e7cd6c2fb1b5b) — Yahoo：WINDHOEK, Namibia, November 09, 2026--Custos Energy (Pty) Ltd. ("Custos" or the "Company") notes the announcement that QatarEnergy has agreed to acquire a 25% p…
- 📰 [Seven OPEC+ Producers Keep December Output Unchanged While Crude Opens Lower](https://finnhub.io/api/news?id=c34103684845d936d59a3f01b8a42a45c00384cdaba695ab2290cd7cd5d754c2) — Yahoo：Crude oil futures were lower on Monday morning. As of 9:47 a.m. ET, the December NYMEX West Texas Intermediate contract was at $87.43 a barrel, down $3.40 or 3.…

**Clorox（CLX +5.57%）**
- 📰 [3 Magnificent High-Yield Dividend Stocks to Buy Near Their 52-Week Lows](https://finnhub.io/api/news?id=bf6f46cb749f9905b093d54661041f2110bac72c59500c1717a93b7997844f62) — Yahoo：High-yield dividend stocks can burn your portfolio, but these blue chip stocks stand out.
- 📰 [How A P/E Ratio Of 4 Can Still Have Big Losses In Store](https://finnhub.io/api/news?id=e93b4fa2870f1851606979796896f5f43e7fd9b5dadff7822395c960015c7e8a) — SeekingAlpha：Low P/E multiples in cyclical sectors like semiconductors, steel, and shipping can be misleading. Read why this may signal peak earnings, not value.
- 📰 [GTA Owner Take-Two, ExxonMobil, Clorox And A Financial Stock On CNBC’s ‘Final Trades’](https://finnhub.io/api/news?id=287474a261c0e091779ded51ed914e01b81c4e7a3889f8661a94fabad1e21a36) — Benzinga：CNBC final trades: Belski picked Take-Two, Simpson Amazon, Lebenthal ExxonMobil and Harrington Clorox, citing upcoming or strong earnings.

**Target Corporation（TGT +4.73%）**
- 📊 JP Morgan 維持：Neutral，目標價 157 → 163
- 📰 [Former Target CEO Brian Cornell on How Exiting Canada Paved the Way for U.S. Growth](https://finnhub.io/api/news?id=83bbb8bcf94033242f2284cd09a8fe513b60387e09d318a687919d050ac6b537) — Yahoo：Former Target CEO Brian Cornell tells WSJ Leadership Institute President Alan Murray why he decided in 2014 to pull the retailer out of Canada after mounting fi…
- 📰 [Illinois is putting communities at risk with prohibition of low-dose THC beverages a month before the federal deadline, says Total Wine & More CEO Ryan Ross](https://finnhub.io/api/news?id=f1ee5ee77a66e8b0b3585712baf30a3a648a68110f749708134f46dec0006444) — Yahoo：On Nov. 12, Illinois plans to pull low-dose, hemp-derived THC-infused beverages from licensed retailers, a month before federal restrictions are scheduled to ta…
- 📰 [Target Trades at a Fraction of Walmart's Valuation. Is the Discount Deserved?](https://finnhub.io/api/news?id=3471c43a2b659db6eea2004dc08315be672d42c05e4c3fe993353c1d35504faa) — Yahoo：Walmart trades at more than 45 times earnings, while Target trades at 17. Here's why the valuation gap isn't as simple as it looks.

**Dollar Tree（DLTR +4.64%）**
- 📰 [PREP Property Group Acquires Promenade at Casa Grande in Growing Phoenix-Tucson Corridor](https://finnhub.io/api/news?id=bd4a09323a22be845bb888a09bc8bc4a70c467a7dcb0cdce43db15254ad668e4) — Yahoo：Promenade at Casa Grande Aerial Image PARK CITY, Utah, Nov. 09, 2026 (GLOBE NEWSWIRE) -- PREP Property Group Inc. ("PREP"), an operator-led retail real estate i…
- 📰 [Unusual Put Options Volume in Walmart Ahead of Earnings Next Week - Is WMT Set to Drop?](https://finnhub.io/api/news?id=1b0fdf75a5216b560e7574fb85e86df0c7e6d1d2a557d715ae87d811bb1ad7cf) — Yahoo：Unusual put options volume in Walmart today may signal bearish sentiment ahead of its upcoming earnings. Other discount-store stocks are up. Is this out-of-the-…

**Hershey Company (The)（HSY -2.58%）**
- 📰 [S&P 500 Top Movers: Gainers and Losers for Nov. 9, 2026](https://finnhub.io/api/news?id=8dbe13d7bad59b65126f092b011cd055233dfd03d70a27cbf9c95faf7c97444b) — ChartMill：S&P 500 top movers for Nov. 9, 2026 include COHR's 17.41% gain, NVDA's 7.27% decline, and moves in MPWR, AMD, FSLR and AXON.
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=20d8a95fb6d88aa6d92d94f3823a12edc8192a1213a79a8e22a78db334a9693f) — ChartMill：S&P 500 top gainers and losers from today's US session, with percentage moves and latest prices for each listed ticker.
- 📰 [3 Beaten-Down Consumer Stocks to Hold for the Next 10 Years](https://finnhub.io/api/news?id=3946d6d23f4d88dd93ca98ebcf7cd52879be40872e4872547506040ed499c53f) — Yahoo：These three brands keep winning customers even as their stocks slump, setting up better prospects for returns.

**Philip Morris International（PM -2.41%）**
- 📰 [S&P 500 Top Movers: Gainers and Losers for Nov. 9, 2026](https://finnhub.io/api/news?id=8dbe13d7bad59b65126f092b011cd055233dfd03d70a27cbf9c95faf7c97444b) — ChartMill：S&P 500 top movers for Nov. 9, 2026 include COHR's 17.41% gain, NVDA's 7.27% decline, and moves in MPWR, AMD, FSLR and AXON.
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=20d8a95fb6d88aa6d92d94f3823a12edc8192a1213a79a8e22a78db334a9693f) — ChartMill：S&P 500 top gainers and losers from today's US session, with percentage moves and latest prices for each listed ticker.
- 📰 [Can Altria's 64.5% Smokeable Margin Sustain Profit Growth Ahead?](https://finnhub.io/api/news?id=fb2e0fc98a4c351cbc31670b7938679fd182acfcd30fecf96a4e4dd105f86605) — Yahoo：MO's smokeable business maintains strong profitability as pricing strength and tax refunds offset declining cigarette volumes and rising manufacturing costs.

**General Mills（GIS -1.77%）**
- 📰 [3 Magnificent High-Yield Dividend Stocks to Buy Near Their 52-Week Lows](https://finnhub.io/api/news?id=bf6f46cb749f9905b093d54661041f2110bac72c59500c1717a93b7997844f62) — Yahoo：High-yield dividend stocks can burn your portfolio, but these blue chip stocks stand out.
- 📰 [General Mills Has Paid Uninterrupted Dividends for 127 Years. Here's the Yearly Payout on $5,000.](https://finnhub.io/api/news?id=77e5ae81cb310d9e667f8f1b1c0b3d0ac2499afe248c0d2f38c69faf80d82f15) — Yahoo：The yield grabs attention, but there's more to consider when evaluating General Mills as an investment.
- 📰 [S&P 500 Opening Gap Movers for Nov. 9, 2026](https://finnhub.io/api/news?id=d9200800176708874dd109c854e6615a2ba6f51cf2b81ae83fce100f0882c5b8) — ChartMill：S&P 500 opening gap movers for Nov. 9, 2026, including gap-up names such as FERG, AMD and NRG, and gap-down names including NVDA, HSY and FSLR.

**Invesco（IVZ +6.76%）**
- 📊 Evercore ISI Group 維持：In-Line，目標價 34 → 33
- 📰 [AMG's Q3 Earnings Beat Estimates on Higher Revenues & Record AUM](https://finnhub.io/api/news?id=7a98d43e408256d7a303620a25a7814c09dcf2a0649c8c428ca10e767c1118bb) — Yahoo：Affiliated Managers' Q3 earnings surged 55.7%, beating estimates as record AUM, higher revenues and strong client inflows drive growth amid rising expenses.
- 📰 [3 Rate Sensitive Financial Stocks As Treasury Yields Reshape Retirement Investing](https://finnhub.io/api/news?id=81588bdd4db211677ebc8bce379730ae097750bc51c29849153f4710046c8a14) — Yahoo：Rising Treasury yields and growing concern about U.S. debt sustainability have pushed long-term rates back to the center of every market conversation, and that…
- 📰 [What Toyota Motor (TM) Said on Its Q2 Earnings Call](https://finnhub.io/api/news?id=8c630b5e407bf52171bac9cd1af8c7813f7aa1eacdcb62e6182cf386a14049af) — Yahoo：Toyota Motor (NYSE:TM) raised its full-year operating income forecast by 500 billion yen to 3.9 trillion yen, citing stronger demand for hybrid vehicles, cost r…

**Franklin Resources（BEN +6.34%）**
- 📊 Keefe, Bruyette & Woods 重申：Market Perform，目標價 37 → 38
- 📊 TD Cowen 維持：Buy，目標價 38 → 44
- 📊 Barclays 維持：Equal-Weight，目標價 34 → 36
- 📰 [S&P 500 Opening Gap Movers for Nov. 9, 2026](https://finnhub.io/api/news?id=d9200800176708874dd109c854e6615a2ba6f51cf2b81ae83fce100f0882c5b8) — ChartMill：S&P 500 opening gap movers for Nov. 9, 2026, including gap-up names such as FERG, AMD and NRG, and gap-down names including NVDA, HSY and FSLR.
- 📰 [11 Stable High-Dividend Payers Are Also Topping The S&P 500](https://finnhub.io/api/news?id=37367a465632a54617e0337baf10656c67b74c36de400a4fc1fd3fb12628ebac) — Yahoo：Investors often think they have a choice: Stable but slow growing S&P 500 stocks or volatile growth bets. But there's a way to get both stable growth.

**Apollo Global Management（APO +4.67%）**
- 📊 Argus Research 維持：Buy，目標價 160 → 140
- 📊 UBS 維持：Buy，目標價 163 → 160
- 📰 [Apollo Prices Offering of Fixed-Rate Resettable Junior Subordinated Notes](https://finnhub.io/api/news?id=9f9532e05bed1e9dcdc97ff14b635ea596a326badc3b289cb735ff3d1f2e8e14) — Yahoo：NEW YORK, Nov. 09, 2026 (GLOBE NEWSWIRE) -- Apollo Global Management, Inc. (NYSE: APO) (the “Issuer” and, together with its consolidated subsidiaries, “Apollo”)…
- 📰 [12% of U.S. hospitals are owned by private equity](https://finnhub.io/api/news?id=6e84efd7dee742cb15d3d93377b9ab1af476d630512ac0497e212d7d8ab6079f) — Yahoo：As scrutiny over cost and quality at private equity-backed hospitals persists, a new tracker shows that PE hospital ownership has expanded nationwide and is con…
- 📰 [Musk “Would Never Lend Money” to Pure AI Clouds. Goldman Is Selling SpaceX’s $40 Billion GPU Debt](https://finnhub.io/api/news?id=49e94e0730c90e527a1596e30df6d253d056b9965ecc3ff422959273c80d34c6) — Yahoo：Key TakeawaysGoldman Sachs and Apollo are marketing slices of a roughly $40 billion debt package that buys Nvidia chips for SpaceX to rent, Bloomberg reported.…

**Capital One（COF -1.63%）**
- 📰 [Berkshire Hathaway May Have Sold More Bank of America Stock in the Third Quarter](https://finnhub.io/api/news?id=0f5d2043b5be50ede7cb1adf3974add60aa2f1a07e6f8ad9d0f8421f4b5a3d9e) — Yahoo：Berkshire’s quarterly filing offers clues to possible Bank of America sales and additional Alphabet purchases ahead of its portfolio disclosure.
- 📰 [Byzfunder to Participate Alongside Leading Fintech Executives and Investors at Exclusive 2026 KBW Fintech Ecosystem Conference](https://finnhub.io/api/news?id=9bc822ccdc36bf2a5519bc8e8dbbe8dc42f083805f92b649b14d5d67f7005b3c) — Yahoo：Byzfunder, a leading technology-driven provider of small business financing, today announced its participation in the 2026 KBW Fintech Ecosystem Conference, tak…

**Brown & Brown（BRO -1.55%）**
- 📰 [4 Insurance Brokerage Stocks to Benefit From Higher Demand](https://finnhub.io/api/news?id=be67b429ec470d4d8cf9cabd28b3d60eda8a1b901e8fee8542874c9fb16e4f9e) — Yahoo：Zacks Insurance Brokerage players like ERIE, WTW, AJG and BRO are likely to benefit from increased demand for insurance products, strategic acquisitions and the…
- 📰 [The Healthcare Cost Crisis is Reshaping Employer Benefits Strategies, Brown & Brown Survey Reveals](https://finnhub.io/api/news?id=2ab021cafc53ef54a18f4d15910b84e62ac7ecc1dec14a645ceb67167346385e) — Yahoo：Rising medical and pharmacy costs driving cost-containment measures as employers balance affordability and employee well-beingDAYTONA BEACH, Fla., Nov. 09, 2026…

**Fidelity National Information Services（FIS -1.36%）**
- 📊 Cantor Fitzgerald 維持：Overweight，目標價 45 → 34
- 📊 Argus Research 降評：Buy → Hold
- 📊 UBS 維持：Neutral，目標價 49 → 32
- 📰 [S&P 500 Opening Gap Movers for Nov. 9, 2026](https://finnhub.io/api/news?id=d9200800176708874dd109c854e6615a2ba6f51cf2b81ae83fce100f0882c5b8) — ChartMill：S&P 500 opening gap movers for Nov. 9, 2026, including gap-up names such as FERG, AMD and NRG, and gap-down names including NVDA, HSY and FSLR.
- 📰 [Cantor Fitzgerald Maintains Overweight on Fidelity National Info, Lowers Price Target to $34](https://finnhub.io/api/news?id=ddf57485e37f16b265eb2e22993a2de8bfe31b08675bf59605e5a5ebc7b1db8f) — Benzinga：Cantor Fitzgerald analyst Ramsey El-Assal maintains Fidelity National Info (NYSE:FIS) with a Overweight and lowers the price target from $45 to $34.
- 📰 [Fidelity National Information Services Consensus Price Target Decreased by 18.33% to $38.82](https://finnhub.io/api/news?id=b1285c6deb446636f37a7115a9fcca6c9ebc8bbee2f00f03e0cc9ca23b9c9ccf) — Fintel


**當日財報｜2026-11-09（S&P 500 成分股，依營收規模排序）**

| 公司 | 時段 | EPS 實際 / 預估 | 營收 實際 / 預估（億美元） | 當日股價 |
|:---|:---|---:|---:|---:|
| Skydance（SKYD） | 盤後 | 0.45 / -0.05（優於預期） | 70.2 / 162.7 | +8.61% |
| Constellation Energy（CEG） | 盤前 | 5.06 / 3.89（優於預期） | 95.5 / 94.7 | +10.46% |
| NRG Energy（NRG） | 盤前 | 4.68 / 3.36（優於預期） | 94.7 / 93.3 | +10.32% |
| Ferguson Enterprises（FERG） | 盤前 | 3.43 / 3.37（優於預期） | 90.3 / 89.1 | +11.22% |
| Occidental Petroleum（OXY） | 盤後 | 1.91 / 1.48（優於預期） | 67.7 / 68.6 | -1.10% |
| KKR & Co.（KKR） | 盤前 | 1.78 / 1.60（優於預期） | 18.6 / 26.2 | +1.67% |
| Take-Two Interactive（TTWO） | 盤後 | 1.00 / 0.96（優於預期） | 16.0 / 16.1 | -3.48% |

盤後公布的財報，當日股價尚未反映，實際數字也可能稍晚才更新。

## 6. Treasury Yield Curve

In [14]:
def treasury_curve():
    frames = []
    for yr in (AS_OF.year, AS_OF.year - 1):          # 年初資料不足兩天時補前一年
        url = ("https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/"
               f"{yr}/all?type=daily_treasury_yield_curve&field_tdr_date_value={yr}&page&_format=csv")
        r = requests.get(url, headers=UA, timeout=TIMEOUT)
        r.raise_for_status()
        frames.append(pd.read_csv(io.StringIO(r.text)))
        if sum(len(f) for f in frames) >= 2:
            break
    df = pd.concat(frames, ignore_index=True)
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
    return df.sort_values("Date", ascending=False).reset_index(drop=True)


def curve_shape(d2, d10):
    """依 2 年、10 年殖利率變動（bp）判斷曲線型態"""
    if abs(d10 - d2) < 1:
        return "大致平行移動（2 年與 10 年變動相近）"
    if d2 <= 0 and d10 <= 0:
        return "牛平（長端降更多）" if d10 < d2 else "牛陡（短端降更多）"
    if d2 >= 0 and d10 >= 0:
        return "熊陡（長端升更多）" if d10 > d2 else "熊平（短端升更多）"
    return "扭轉變平（短升長降）" if d2 > 0 else "扭轉變陡（短降長升）"


def fred_curve():
    """備援：FRED 的公債殖利率（資料同樣來自財政部）"""
    if not FRED_API_KEY:
        raise ValueError("未設定 FRED_API_KEY")
    cols = {"3 Mo": "DGS3MO", "2 Yr": "DGS2", "5 Yr": "DGS5", "10 Yr": "DGS10", "30 Yr": "DGS30"}
    data = {}
    for t, sid in cols.items():
        r = requests.get("https://api.stlouisfed.org/fred/series/observations", headers=UA, timeout=TIMEOUT,
                         params={"series_id": sid, "api_key": FRED_API_KEY, "file_type": "json",
                                 "observation_start": f"{AS_OF - timedelta(days=30):%Y-%m-%d}"})
        r.raise_for_status()
        data[t] = pd.to_numeric(pd.Series({o["date"]: o["value"] for o in r.json()["observations"]}), errors="coerce")
    df = pd.DataFrame(data).dropna()
    df.index = pd.to_datetime(df.index)
    return df.rename_axis("Date").reset_index().sort_values("Date", ascending=False).reset_index(drop=True)


try:
    tc, source = treasury_curve(), "美國財政部"
except Exception as e:
    print(f"財政部資料無法取得（{safe_err(e)}），改用 FRED")
    tc, source = fred_curve(), "FRED"
cur, prev = tc.iloc[0], tc.iloc[1]
rows = ["| 天期 | 殖利率 | 變動 |", "|:---|---:|---:|"]
for t in ["3 Mo", "2 Yr", "5 Yr", "10 Yr", "30 Yr"]:
    rows.append(f"| {t} | {cur[t]:.2f}% | {(cur[t] - prev[t]) * 100:+.0f} bps |")
d2, d10 = (cur["2 Yr"] - prev["2 Yr"]) * 100, (cur["10 Yr"] - prev["10 Yr"]) * 100
spread, spread_prev = (cur["10 Yr"] - cur["2 Yr"]) * 100, (prev["10 Yr"] - prev["2 Yr"]) * 100
display(Markdown(f"**美債殖利率｜{cur['Date']:%Y-%m-%d}（vs {prev['Date']:%m/%d}，{source}）**\n\n" + "\n".join(rows) +
                 f"\n\n- 2s10s 利差：{spread:+.0f} bps（前一日 {spread_prev:+.0f} bps）\n- 曲線型態：{curve_shape(d2, d10)}"))

**美債殖利率｜2026-11-09（vs 11/06，美國財政部）**

| 天期 | 殖利率 | 變動 |
|:---|---:|---:|
| 3 Mo | 4.24% | +5 bps |
| 2 Yr | 4.56% | +5 bps |
| 5 Yr | 4.67% | +3 bps |
| 10 Yr | 4.88% | +2 bps |
| 30 Yr | 5.26% | +0 bps |

- 2s10s 利差：+32 bps（前一日 +35 bps）
- 曲線型態：熊平（短端升更多）

## 7. Free news

In [15]:
CNBC = "https://www.cnbc.com/id/{}/device/rss/rss.html"
PAYWALLED = ("reuters.com", "bloomberg.com", "wsj.com", "ft.com", "barrons.com", "marketwatch.com",
             "economist.com", "nytimes.com", "investors.com", "washingtonpost.com")


def google_news(query):
    return ("https://news.google.com/rss/search?q=" + quote_plus(f"{query} when:1d")
            + "&hl=en-US&gl=US&ceid=US:en")


GROUPS = {   # (來源名稱, 網址)
    "盤勢與個股": [("CNBC", CNBC.format(100003114)), ("CNBC Earnings", CNBC.format(15839135)),
                   ("CNBC Investing", CNBC.format(15839069)), ("Google News", google_news('"stock market today"')),
                   ("Finnhub", "finnhub:general")],
    "總經與聯準會": [("CNBC Economy", CNBC.format(20910258)),
                     ("Fed 新聞稿", "https://www.federalreserve.gov/feeds/press_all.xml"),
                     ("Fed 演講", "https://www.federalreserve.gov/feeds/speeches.xml")],
    "美債與美元": [("Google News", google_news('"Treasury yields"')), ("Google News", google_news("dollar forex"))],
    "黃金": [("Google News", google_news('"gold prices"'))],
    "原油": [("CNBC Energy", CNBC.format(19836768)), ("OilPrice", "https://oilprice.com/rss/main"),
             ("Google News", google_news('"oil prices"'))],
    "中文（鉅亨網美股）": [("鉅亨網", "cnyes:us_stock")],
}


def fetch_items(name, url):
    if url == "finnhub:general":
        if not FINNHUB_API_KEY:
            return []
        return [(datetime.fromtimestamp(d["datetime"], timezone.utc), clean(d.get("headline"), 300), d.get("url", ""),
                 f"Finnhub / {d.get('source', '')}", clean(d.get("summary")))
                for d in finnhub("news", category="general")
                if d.get("datetime") and not any(p in d.get("url", "") for p in PAYWALLED)]
    if url.startswith("cnyes:"):
        r = requests.get(f"https://api.cnyes.com/media/api/v1/newslist/category/{url[6:]}",
                         params={"limit": 50}, headers=UA, timeout=TIMEOUT)
        r.raise_for_status()
        return [(datetime.fromtimestamp(d["publishAt"], timezone.utc), clean(d.get("title")),
                 f"https://news.cnyes.com/news/id/{d['newsId']}", name, clean(d.get("summary")))
                for d in r.json()["items"]["data"]]
    r = requests.get(url, headers=UA, timeout=TIMEOUT)
    r.raise_for_status()
    items = []
    for e in feedparser.parse(r.content).entries:
        t = e.get("published_parsed") or e.get("updated_parsed")
        src = e.get("source") or {}
        if src and any(p in src.get("href", "") for p in PAYWALLED):
            continue                                   # 略過付費牆媒體
        title, publisher = clean(e.get("title"), 300), src.get("title", "")
        if publisher and title.endswith(" - " + publisher):
            title = title[: -(len(publisher) + 3)]
        items.append((datetime(*t[:6], tzinfo=timezone.utc) if t else None, title, e.get("link", ""),
                      f"{name} / {publisher}" if publisher else name, "" if publisher else clean(e.get("summary"))))
    return items


def group_news(sources):
    cutoff = datetime.now(timezone.utc) - timedelta(hours=NEWS_HOURS)
    seen, items, errs = set(), [], []
    for name, url in sources:
        try:
            for it in fetch_items(name, url):
                key = re.sub(r"\W+", "", it[1].lower())
                if it[0] and it[0] >= cutoff and it[1] and key not in seen:
                    seen.add(key)
                    items.append(it)
        except Exception as e:
            errs.append(f"{name}：{safe_err(e)}")
    items.sort(key=lambda x: x[0], reverse=True)
    return items[:NEWS_PER_GROUP], errs


with ThreadPoolExecutor(max_workers=6) as ex:
    results = dict(zip(GROUPS, ex.map(group_news, GROUPS.values())))
lines = []
for g, (items, errs) in results.items():
    lines += [f"### {g}"]
    for when, title, link, src, summary in items:
        title = title.replace("[", "(").replace("]", ")")
        lines.append(f"- `{when.astimezone(TW):%m/%d %H:%M}` [{title}]({link}) — *{src}*" + (f"：{summary}" if summary else ""))
    if not items:
        lines.append("- （無）")
    if errs:
        lines.append(f"- ⚠️ 無法取得：{'、'.join(errs)}")
    lines.append("")
display(Markdown("\n".join(lines)))

### 盤勢與個股
- `11/10 10:55` [China exports, imports soar past expectations in October — surplus narrows sharply](https://www.cnbc.com/2026/11/10/china-trade-october-exports-imports-surplus-.html) — *CNBC*：Goods exports soared by 35.2% last month from a year earlier, exceeding expectations for a 33.5% jump in a Reuters poll and slowing from a 36.3% jump in September.
- `11/10 10:21` [Is Stock Market Open Or Closed Today For Diwali-Balipratipada? Know NSE, BSE Holidays This Year](https://news.google.com/rss/articles/CBMi4AFBVV95cUxQWURCZzFVU0xiakpQVGlhWWZHZ1FQMG5NWVYwODlQc0V4TE15b0R1X1ZHUFZNdU5SNGxOSGZoQUd2MlcwS0JLTGFiN081RTBtV1E3QTd2MjQwczBvSTlYSTFzSFhtRk01QkNsZmdCYVozemU1ZHUtcjJENGlJMngwZWNvc0gxN1hEQWpFcThNYVIxY2pkeThHaUFyOW1KcC1pTUxQeHRSc1hjdm0xZ0V0ZUpDTm82NGdHeVZYUGtKZXhFakNYa1RtVk04bWtXQ1A2V0UtNmdQSi1rV0RCRVpDcNIB5gFBVV95cUxOZVdxTXhOcm9mMUF1NnhqeDVPVExvejJRY0IzbUJ3al9GX0dVektzRHRRUDVhSWdRX0pMNU00WHRtam1pcTVTVlh6amFPSnE1c2pRWnF0RXhhSGREcjhJQmMxdWYxVzNaU0p6WDJldGcwYlBoYTdfVW5aMDk5a0s5R2p1bUNfemlWSGtZS2puOTV5UEZxVXRBLVJGSW40d2duNndsYWVQTnB2bGVMc0JEZmo2NzhkUXRpd2xmRW5oLTlwXzNUcjFsbWhZODFuZEdndWtZYXk1Zk5DQ0pPUWkzeTNYbkQ3Zw?oc=5) — *Google News / News18*
- `11/10 10:17` [Stock market holiday: Closed today on Tuesday 10 November 2026 on Diwali-Balipratipada; no trading in NSE, BSE exchanges](https://news.google.com/rss/articles/CBMilgJBVV95cUxOU0FvTVExZkl6clFNWm5rSFk5NVhWV2szaW5SUEtpOGh3YWFpcFVQaU9CcHJZUlMyMmt5TXBkamRzSC15b2RmVk1WR2Y3ZXZEMURjZ3dnR21PTGw3Z1ZaTERlZGt1TFZJR1U5d2FNY1VHTGxINTh1dXFjWElvcFY1SkhRVkYzZ3NCbXpLRU5INXA2dF95dHQyU2lhNl9CRmg0eEg2cGJHYTM4eDVwTGpYS3I1WVFrcnNQdmtaQUo0Y3dYNFdpNTJVQVcwelRNMExYRzRId0p4SHlnTXR6YWFlamE4YlBJWElSLXVFbGxGZ3RCdDkteDBIVUxpNFVobWhJWVd6MVZzN1ZyVDA2TlFPT2NXRGxwQdIBmwJBVV95cUxQNjJUS2RpckxQMlhGcnkwUE5GMWUwY3JFakF1bXRobmxpZW1UX1BtcGd3dEEtb0EyclpJRDdNYlZuTHVGdGNVbDVsVE00RHhuOWkyLUVOSUQ4MzhUeG8ybk1QSUJwT0xPREJBQXZzZWRYd0xLeldJaXdaeGkwdFJvMXVEU3ZQTTZzNjctcmowMzJrM3g5RnR3NS1WRzFOMDVzdWF6MFFhVG9ESHdwTHpwMnNTWmdSV3lvYnZ3ZEU4UzJycHBXaW1OUTZDbzBrNFZuMnctZUJtQTlJYmoyOUJUMmEtM09FZXNtQ1RTbjhNczIzc05oRWc4bE9LU3FINGlST1RlRW5KV09VaGNQMldOLUVyN0ZmYUIxLUhN?oc=5) — *Google News / Livemint*
- `11/10 10:17` [Stock market holiday: NSE, BSE to remain closed for Diwali-Balipratipada](https://news.google.com/rss/articles/CBMi2wFBVV95cUxNVlpPWlh5emZPQm4xdDRERnV1bjNyUUlBekZ5cUxmZFhIc0tOU0E3ak9DLUdPVnVxYnFnN0JoYVdTcEstb3ZTZ1ZQRkZtREtSd0tOelhRR1FUVnpOTlpnX3BsZkZmeHYwVVphSTdld0dMaWNHMy1KYnJjM3ptZmhRMUh1aDRXLTl4ZkJELWZlTnpwanJlekItRzlvMk53ME9Ka3NhSHFpUEdSc2ZtY1dCc19MWC1BeGRBLTJ3bkotN2cwVEZrRVphcVR1RkJIcll3ajBGUEdObWpxNlnSAeABQVVfeXFMT29XcVZicVE3ajlJd19CU29CR0RhbGhJckNTQnF2MUl0U1ZmYjdQZ2xUdWx1cE1MUEZSd3JJQ2RXY2lUMWN2RlczdnVfSi1VTVRFQWRzYWNCYUx0SEc5aGRrSEgybDItZDBEMS1Nbmcyems2Y3VYQUxudk9VSmxuYkJzLUFWMWtlN1JYVnh5d0k1eUt2TkpFZmJiRkNpZjJ4dXlkSmdoSVlaUy0wdHprNU9QOWktZGl5aERMOHRYWDB0bWdPLWpHNWN0RGljZmNOM2ZmQUhzc05YcjBCUGRvemg?oc=5) — *Google News / Business Standard*
- `11/10 09:50` [SoftBank Group shares surge 11% as Asian tech stocks track Wall Street rebound](https://www.cnbc.com/2026/11/10/ai-softbank-sk-hynix-samsung-wall-street-tech-stocks.html) — *CNBC*：A sharp decline in oil prices lifted sentiment while renewed buying in semiconductor names overnight spilled over into Asia, following last week's bruising sell-off.
- `11/10 09:43` [Trump says Jensen Huang told him Monday he didn't know paramour 'was a spy'](https://www.cnbc.com/2026/11/09/trump-jensen-huang-nvidia-chinese-spy.html) — *CNBC*：President Donald Trump suggested he had only recently learned of allegations that Alice Huang was a Chinese spy.
- `11/10 09:30` [Stock Market Holiday 2026: Are NSE, BSE closed today on account of Diwali-Balipratipada? Check remaining](https://news.google.com/rss/articles/CBMiowJBVV95cUxPYzM4V0FETUpjMnB5aUpDQXQwZjJlUWRJbmZJVU9hYUZkZmptMXZha2h4TmFaZ3JNYWhyYVFRZHFTRUZfV1pWQ3N6UVc2WVFWNGFEdzJhS21iemJKSC16Xy1PVFlrenhtS3VEMF9vRzg2VHdPQ21wSUU5TEt5ZXdiQ2JtVWVheTJ3d2JiY19PSlpZcWN0R3k2UHJ1UnFCZ0EzOVdGeHFQUE5NNW1RdzhWZ3J1dXZwZTl6a2psVzZhdEZSMWNWOTZQVERzSmZzcXhNWHU5bDV2Z0pQQjJmTmN0el8ycHFNNWtfVUR4QkRuYUlTZG5LRW1LZjg3NUx5Nnd3TlVKSFRuTGR3ZjdyTkpVTTUwd3FJNWFBN2hGNGEtY0t5Y2_SAagCQVVfeXFMTm1IcTdFMzR4Sl90NDcya2J2U1dFb3hiZW94WXRFYmtkMnFEbE5BMk1vU3N0ZjF3dVlVTnJuVmJhTTljSDZNQzlEODRIT1lIRU94YmlFSUdNbGREREhaMTVLUTNVSFJtTEZkWjhRN1N2bVFYSExNMmdDYWxQeDI5Mm1SMU9IYTBwc3Y1VGRqRlpBQ201VExNbEFWNFF0LUwzcTF1b1FrREZJalE2aEZpSXdsTjV5dGphMGUyakttai11Y094dVRPSVdIQ1N0UkhoQjRvY05uRDBIOU1adk8xTEg1cE44TXJPeUU3Z3RFUVhMaTNHalZUdkpRLS14TWJVQ3B6emx5NkVMRmFUNkJlUHpkOVJ5b1ZhUXMxNmhualBRV2ZXaS1TYmc?oc=5) — *Google News / The Economic Times*
- `11/10 09:06` [Asia Markets Today: Nikkei 225 Surges and ASX 200 Climbs After Wall Street’s 2.35% Jump](https://news.google.com/rss/articles/CBMiwAFBVV95cUxOR3ZwME5pdFNfSzNfd2V4R0NWZFIxXzZ2TEtRQms2dEZKODduLTJMMEJSUmtsOW42NGZaQnZQd09GbFZNdWhlSmxrMHVvT3pZRUtvOGlXQlZ1ZUtNblJqdFpnOWd1a3Itc0E4NjUxSG94YmZjV21rNEU3bExtZDRPRUdyRVNIRE83bkI1UzZXZHE5UUlBNkdkV3gzVXl1cVJvZkNOeDRpd0J5QU1zeWhwZm5pSV9ZaGNRNEZzeWRvRzQ?oc=5) — *Google News / JournalArta*

### 總經與聯準會
- `11/10 10:55` [China exports, imports soar past expectations in October — surplus narrows sharply](https://www.cnbc.com/2026/11/10/china-trade-october-exports-imports-surplus-.html) — *CNBC Economy*：Goods exports soared by 35.2% last month from a year earlier, exceeding expectations for a 33.5% jump in a Reuters poll and slowing from a 36.3% jump in September.
- `11/10 06:23` [3 unexpected reasons the October jobs report was so strong](https://www.cnbc.com/2026/11/09/october-jobs-report-hiring-by-sector.html) — *CNBC Economy*：October's jobs report blew past economist forecasts. Here are three reasons.
- `11/10 05:03` [A key report for where inflation stood in October comes out Tuesday. Here's what to expect](https://www.cnbc.com/2026/11/09/october-cpi-report-2026-inflation.html) — *CNBC Economy*：The CPI headline reading is expected to show an increase of 0.3% while the core is forecast to increase 0.2%.
- `11/10 05:00` [Federal Reserve Board announces approval of two applications by Mutual Federal Bancorp, MHC, and MFB Bancorp, Inc.](https://www.federalreserve.gov/newsevents/pressreleases/orders20261109a.htm) — *Fed 新聞稿*：Federal Reserve Board announces approval of two applications by Mutual Federal Bancorp, MHC, and MFB Bancorp, Inc.

### 美債與美元
- `11/10 11:00` [Australian Dollar remains subdued as China’s Trade Surplus eases in October](https://news.google.com/rss/articles/CBMiuAFBVV95cUxNQ0RFMzN6cHRsaklxeGRtME14WWFBRkpSQWJnTHZKUnd3VXdsYlc0MEFONHVZTzFGVm9qRWFISERjUElWZGwtdE9HVFZkQ2lyVkc2LTd4QXlxSEsxMGJnRHk3azFOSjVObmxQcjk4eU5zUU1uaklkMnlUS2RhN2lOOVdWSkV3OVlfZzlMTzJOTzh1VDc1c2NmcG1jX3dTS1pwOXZjNnhOTUk3a0ZEeXhPMXN4LWg1SFFO?oc=5) — *Google News / FXStreet*
- `11/10 10:14` [3 Bank Stocks Tied To Treasury Yields Investors May Want To Watch](https://news.google.com/rss/articles/CBMiggFBVV95cUxORVgzZndnYVBGMElfd1Z0dTZpRUlIb1ZaTTVxQUN1aDV4azh3YWhyMGlwSXhBYl93Rnl2ZnpoWlh2MUpmN0hXWFhTX0o4c0dudWJJU0JYajhyakpxQU5rOXpoam1ESmpEUDRJSGZTTmZ5RXpEYjJqd0RncjFQMjNJRHpB?oc=5) — *Google News / Yahoo Finance Singapore*
- `11/10 09:59` [Treasury yields fall across all maturities in October on foreign futures buying and safe-haven demand](https://news.google.com/rss/articles/CBMibEFVX3lxTE05OXg1X3hwdFo5VjRhdERHVk93c1hSNnR6OFBEaFlMV0E4YTNkdkEwQjBVM3RZRHdDdGFZaHFfZ1pEUHVGV3FhZHoxZnpVaUFBOEJWYmVhdG14MkhrY3Y1eDBiWHdHbUFPekdKWQ?oc=5) — *Google News / 머니투데이*
- `11/10 09:58` [Foreign exchange rates, USD/VND exchange rate today, November 10th: The attractiveness of the US dollar as a safe-haven asset is diminishing.](https://news.google.com/rss/articles/CBMi1gFBVV95cUxPVXc2OWdDNUVMZ09qdGJyUkY0LTRDclByTUZuQnZXWjBOVzIyRUdSdHVmT3YzbEkyamtSVlJvcktwdjN4Q040ODBMQUNkd1RTSnFqblBlX3RVZ0J3NGhxTmZzdFZDNzM1WHBlRnRtNE4zTEl1UnJ2QUxUZFdaY09xOW9XZS10UHM1cEpUaFZnTEN4VXVQMnN2MVJrSTgwb0F4SVlycklwV1k1bXA4dk5RbFBXOXM5Q2hwRjJ6VFRLdWxKaVFQOGpCWFNLbnlLTDNTWXdySlJR?oc=5) — *Google News / Vietnam.vn*
- `11/10 09:19` [Goldman Sachs Treasury Yields Outlook: G4 Yields Fall 25–50 Basis Points](https://news.google.com/rss/articles/CBMiU0FVX3lxTFBKYkhQaUZ1MkZNaENjcXZ4Mmp1TW8yNFJRSHhqTzB2RDFWN3BOODJ4N01EQXZsWlpqc09HYVo5Y1Y5RG5HOXpMVjU5bmF2T3ZuSzh3?oc=5) — *Google News / شبكة تواصل الإخبارية*
- `11/10 09:08` [Rupiah is Predicted to Weaken, Investors Beware of the US Dollar and Geopolitical Turbulence](https://news.google.com/rss/articles/CBMiR0FVX3lxTE0xekp0VWhXcVZYUHdTNHJEYXB1cjBNajV4REN6cldRNFJIX2d2dm9oMDY5Z0JoV0NHTWRiaTJLSEd3TlgzWXV30gFCQVVfeXFMTldScy10NHpUclRFRzVla0tKNEktNmlTYlVUejYwZFdFYkRObWhJX01BU1BZVzhySG9tT3pGYk00cE5n?oc=5) — *Google News / VOI.ID*
- `11/10 09:04` [Ethereum staking yields about 3%, below 10-year Treasury's 4.8%, raising questions on its worth for investors.](https://news.google.com/rss/articles/CBMikwFBVV95cUxPTlJPUDE1S0t4NHI0WnRyQVNfTElqYWZBbGI5OHJ1U3NUWFlRSjd3alpHenlZcXVLSzZkeW9menhHNWprWWZyZHdCZ1UwRkw4LTNNWVRBRHV6aXhITjQtVS1KTy1COWZxeFh1Mndnb0k1dkVSaHF0cFVKaDFNeVlFemd4V0Q3bHZ6bnJtNVBoSUVGWVE?oc=5) — *Google News / Pluang*
- `11/10 08:51` [Japanese Yen Outlook: EUR/JPY and GBP/JPY Outbid USD/JPY](https://news.google.com/rss/articles/CBMioAFBVV95cUxQNm5WcmppbGlzeE1tbzBHQXRhVWpZd3dVMmJxVHIyd2JqQktnYU1zalRXS2N2ZkJDdExQVThkaTRsRXJ4M21BZkQ0RVBaU1BscEtjeDYtYXN0dDRSajNFNWxZUmREdUYyMkxXQWY2SFZZSXpxXzA5SXRtWG1fcWJpM1JJek9qYlpNNGpaSXo2NTJPdURmSzZncVExeGhRb0I2?oc=5) — *Google News / FOREX.com*

### 黃金
- `11/10 11:29` [Silver prices lose momentum ahead of US inflation test](https://news.google.com/rss/articles/CBMingFBVV95cUxQcExpVmR0Q282WEFoNWZQY1Y3STl2aHVMcWhjRl9kelNOVnRDcVAxdDJsNWFsb18tdkY5VE9XYVdkd1l3Z3lnTGIxVVZkcXFkQlZIc2xuQUVHeFRhejdnaVRyc2NiS281eUs4QXU5TTBIN1ZSTVNtMnhxei0tUllnTmJaaWY3eFRQOWEya2RXUjlyVHhIYUR0b1hYZWtoUQ?oc=5) — *Google News / Laodong.vn*
- `11/10 11:09` [Today's gold price update (November 10th) suggests it may not have bottomed out yet, but the market still has long-term support factors. At what price range should you buy?](https://news.google.com/rss/articles/CBMi3AFBVV95cUxPQlViVndjNnhOX3I0aV93Z2FWVF95VjlhbDE5UEd0Si14LWpqNFFDMW9jYU9ldFVmTjZPYzdXblhxVmMtbklFUEdUSGhjM3M1OGtoaXFpNjdLTHhzYWZseFFYdkhaY2NVa0hoZEZRQkFVTzdmbjhpeDJ5Q3cxUHhic2ZySzM3dWtTNmItdjlCcHNLcE1PSHI2eWhEMExieVVXdk84YTd0Mm9qMzNINEhFSjEzRlFJRXY0TkZxcm9CSzU1Z2lSdlNLVC1Ld1EycHNnR252emxWSE9kb2Np?oc=5) — *Google News / Vietnam.vn*
- `11/10 10:59` [Heraeus reveals noteworthy developments in the world gold price market](https://news.google.com/rss/articles/CBMiugFBVV95cUxPNmJVWWtiLVlxSnU3ek4xUVQ5OWtwUHIyUzlzMUZNdlNFVHlmU3FhVjQ4TmFmSmZHRmYzT3J4XzZrWjFLTVhDalFQblpUZEN3MUdIUDZ5akZ2VTc3cjZMV1E5SGtaRjM3dDl5SndiU2hkZklWa21oVGdsZzJyT2RoMkVudmZhdVo0ejBjMDExRUc3QmlDYmdlM2U5VXRIMFFFUUdsRlpLbnVZbkhPODZ1ZkVZdGNfT1lJemc?oc=5) — *Google News / Laodong.vn*
- `11/10 10:57` [Gold price weakens, buyers lose up to 7 million VND/tael after 3 months](https://news.google.com/rss/articles/CBMitAFBVV95cUxQa1h4dmtUTFpzXzZ1WGUxcGFSWV94MmpweWV2enZRTXNNNE9YcU90S3Y4NlYweE9Ycm9ISkFRSGVPUnNyVExvQTE3SlBQZjdIUTA1Tm5FN1hYejd5MldoZkg0VWV0YURidVJvODl2MGJlT2N0OTJJbTdOdk9mZVowRGJtV0ZFQ1FERnZsUG54czZoODA1aU1lcTJOdTUwaW9xaEsyYzlnQUh3WVJVMnlfVnRpR3o?oc=5) — *Google News / Laodong.vn*
- `11/10 10:26` [Gold and silver price today, November 10: Check rates of 24K, 22K gold in Delhi, Mumbai, Kolkata other cities](https://news.google.com/rss/articles/CBMiigJBVV95cUxOc1F0T05SRi1sa005M0ljYnkyR09acTVRQkxDUHFCN09jRm5oaGM3YWhaTGhaLW55aklabnE2aXFRbTBnOW5sWGwxZDlET29DeHd2c2FPenRmeC1adExPTy1abFBMTXJ0VzMxdjVVcFE4aDUxb3dlaUxmNW5JTjRjZlRmTXRUOUh1V2hBcmFtMDZPcTFHbkE4ZGVCU29Nc1V4djFPT1ZMb3BVcktxeFhHcVFoM0NQYzhHdlJnRlY1Z0FVUWU1THkyaFNyMVRRX01oSkJSYUV4S0VXS05SQjlxcUhkR0g0VDVLcV8tYkgyY0ltSFFVRUtVMGlnRkhvWXB3SEV1QS05N05JQQ?oc=5) — *Google News / Moneycontrol.com*
- `11/10 10:18` [Gold and foreign currency prices 10.11: Gold rebounds, USD weakens](https://news.google.com/rss/articles/CBMiqgFBVV95cUxQa28yNXdseGtpcnZhWXZMc0ZpX3Z3c2VMVmpuWEp0NFRXTWJ6SkpKSkx0cjVTZFhnRFhpWVBNRU5tUWFrNVdWWFFmeDJmQkhRNEY3Y3FscmlpYzlfcGxhS0xqZlRRSXNYNmhLVTdsUjU1YTlpMk1DRjJ5X01JOFZ2U3J0aE1LZms3SG5OckNXNUI4bnlSb2x5TzNQVHh2Wjc2dlZ0SGp4Q1h5UQ?oc=5) — *Google News / Laodong.vn*
- `11/10 10:17` [Gold price today, November 10th: Gold rings fall, gold bars rise in price](https://news.google.com/rss/articles/CBMisAFBVV95cUxPUG5jQkdKQmlkeDVJTWpBYTdkc2VnRXNmLXllcEpqejdFS1duMkplQVB4d05oUjRDYjg4dmtBeWJGQTk4NXozanVkRURLRE5vemdWUThmaXAwa0paNlJTdG1vaUJUZURGZ1dqSW1vbjJ5a3dRazFoN2RramUxTlRXX0xOX0hRcFczZzBhWHZVSjhwOEhlT2NKd0J5Z0NIMkM4SG9jekNKaEV0RFRGcVJzNw?oc=5) — *Google News / Laodong.vn*
- `11/10 09:57` [Gold Rate Today: Check 24K, 22K Gold Prices in Mumbai, Delhi, Chennai, Kolkata, Bengaluru on November 10](https://news.google.com/rss/articles/CBMi2gFBVV95cUxOZ2ZvbGViQkNqWGhmUTVGcTFScUZGUXhaUEI4RlQ4d2l6TE5pMFkxYjVFN3JaVDIwSzVodGx4MEQzY29nTkFzVjM0eHVuUWloTzY1MkRoWGtEYS1rN1dWc0xDR0xjS2tsRi1oa2Jrd0VVVEV1eEpna3lpVlJmVzdzcHlzZjBGUXZlQzdZOExHcHlSUVVBVFF0a2pRQjEwaE01V2dueEtUR29CbVR0SDEtd1NPUEgyYUZRWDcxM3RKdlRHdF8yT2NKUFdmelpwdEFWSzZaTF9KMlVKZ9IB4gFBVV95cUxQYWJUVm8xSnRNaTdlOGh5SGNYcUJOSFlZMXN3YTB1eVhRWDYtX0VtdGJ2cmFLZzY5YjJtb21WRXJTMGwxYmUwSU9TVWdGbkpKQVcyVGJTOU1NLWJEUUo0LW1ORVhxMzhwcTEtRFkxWHF2MWFlbXMxbVViemNlNWh0NTNiYzd2SGNLMlc0WHNqeDE5QkpuOTJpY0piRlBlbXN3N0RZODFLU1VacW51MktQbmZRREZGSFUyTlRzMnUxNC1DS25TRXlaSS1iMm5jRHlNLW5hNXhNeHpCdTcyVG1GWUR3?oc=5) — *Google News / NDTV Profit*

### 原油
- `11/10 11:05` [Oil prices extend decline as Trump softens stance on Iran, easing Middle East risk](https://news.google.com/rss/articles/CBMidkFVX3lxTE0zTURwWkRUVC1yMS05TnRCeDRkZ3lGd0xwNXhING9MUGhVT0FzM2RuVUF5YUhkTXFRLS1LTWU2cElkMTdDWUs0WE03ejFWVjIzUVBFLWZFNWJzM3J2bjB6OGt6WmRNY2ZmQm5POU5SQXJFWVgxeEE?oc=5) — *Google News / BigGo Finance*
- `11/10 11:00` [Wall Street jumps as oil slide cools inflation worries; Nvidia slumps](https://news.google.com/rss/articles/CBMijAFBVV95cUxPWlJJUExTNmVJUFZVM3dfWVBCdnN5OGZVRnp4S3IyTVVVT25oOFJuamh2MF9aVXh0UzRnUGdfN1NzOExUa0Q1M05GenAzSDU2R1BHNVRfOFBTdHQxM2RoQnZuRC03WHY1WVU0b01BbndaRWpaX0ljVkxfa0RpdGFkUWJuSFdOd0VRdDlUOA?oc=5) — *Google News / Virginia Business*
- `11/10 10:46` [With peace expectations weighing heavily, how much further can oil prices fall after hitting their lowest level since August?](https://news.google.com/rss/articles/CBMirgFBVV95cUxOSGRyMTF4c3lJc2lwS2pkS3R2c3B5RGZ0N2E0TjMyM0pBZnlWQzRLWDgwQkZuVzdGVFhfczN6Tm5JY25pZnlZZWlXSE5YQ1VsNFZIUGdkMEZkbGFjVWQxaDNvYWkwNWl2bFdBaGNXbjdFUmZHejNzSkw2TXJiTE16WU5XWDNjZ2I0bzdfUi1vdlNmSTVRYk9kWmFoZ2oxSjZfUnlZcE5IbzdFb0J2VFE?oc=5) — *Google News / Moomoo*
- `11/10 10:29` [What to know about fuel prices after a midterm elections oil price drop](https://news.google.com/rss/articles/CBMinwFBVV95cUxOQjBqc2NzTk9HZlUzYkl3QTg3Mmhha1I4X05sMzYxSTliaW85V0hTY0RJZWswRlZLT3FBRjlNZVJQX3J5d0dFWDh1Q3NXZi1jd3VrU050bXp1T2Frci1CQWtGWFY5QmFaNjcxY0loV1VSREc5ZU1OWUdURzE2Y3dQd2JEc1dQSkhTS25Ic2NuNkdUU0YzQXVtVUZJVnhjSGc?oc=5) — *Google News / ABC News & Headlines – Australian Broadcasting Corporation*
- `11/10 10:26` [Oil prices today: Crude edges lower as Donald Trump signals pause in military action against Iran](https://news.google.com/rss/articles/CBMimgJBVV95cUxQaWVwMWNid3F1N0xoQmZqRnNYSjVEX25Db2dnVFdHRHlBR1VscmtyNXVBS2w5eE91eFZoVUlFS0hYckN3THVSRWJSZFo5dVN2clpXWGN2a3haZWR3Sy1BWmM3X0pDQnFPSEhfOHFVcVlBTGtCSkI0SHBZejhiWVdvb05yLWFXbmJTZWd5ejhiZnBESWVWX2tNNjAtM2hhcmQ4MkUwZDR4NThYbEtFN1RPeWhxWXBXTjVXOEJ6Sl9qM3lfNXhWLUJwemw3dURGakl5S3VFdkRIcU9SUlNxLVd0ZWo5aF9Qc05ONnptbjhIRjBESElHdHZVSXdvQTlrQjNsczNFVzJDcGZxbG45VWRzMTFnMXN3Z1B0SXfSAZ8CQVVfeXFMT0NQNzRZeFEySzBVYkJrV2JCU3hRZUNzaXlSZ0NJQmxaYlhjb3oyNkI1cElTcU5UTHBCcmw5LTdTbVNJcW9KMU1SVzVLN3BGWDI5Q2d5Q1VONlMxRFIxbE1UX01YcXMyRVZnVjgyOW83N2toTVJlOGVNVXQ4SjVnMVR4NEp6MFZTUFB1eXpacHhIS0U1cHpPSXlJVWswaGhNVFEtTVE3bHpCQzFjSWVQUGhoRjVPQ2Q4SmVma0JOMFZtVWNmVm1OVkIyREVLUk9aMlNneU51OWN6NzdkUnV2d3NQd3BsRUxFbFctZkZYN21yLUdWMHJsaEoxeUVnMHYtT1B0TzlmZmNNZ3ZnNGswbEJwcURnZHduZVRzamppR2c?oc=5) — *Google News / The Times of India*
- `11/10 10:23` [Asian stocks climb as sliding oil prices buoy sentiment](https://news.google.com/rss/articles/CBMi6wFBVV95cUxNS1RFMk5uVG1uZTdfQjJJS0hqZGpNeHVJQVl1a3VvUllZRVhGWHYxc1I1NFpDSzZqSnlTWTN4UVE5R0lDZXVfV2gtYmVpTWYxZDBHTGZDMlgzNjA0ODlUc1VSNFN1RWI5R29fUGFKajl2ajczdHNOUWEtZnFRbWxoV3ZDV3F0N2VoU3pmVUg2OXJTMXFvZXJuWERJTVNUVjZTTmNseGtnV3hJUHlZS2h5N0lySTRlTWVIelM2VUw2N0RUNHhoak1vMW16cVl2SzRlajRfSVpKRzVBeEV4YnlDb1hYRnM2R19UUURJ?oc=5) — *Google News / The Lufkin Daily News*
- `11/10 10:07` [ASX Jumps as Oil Prices Fall on Trump Iran Remarks](https://news.google.com/rss/articles/CBMihgFBVV95cUxNNzRqNnJkRHVzZTEtaVJYaGxMNEFRYUtlRmtiR1RLeWVtU1UxeUx5NEdUeDl3VG5acXhKWFVpNkgzUTBsMzhrbm9uQWdGRWJ1S3psbjFrZ291dl9qc2pWRHdHVXEyTmZEQzBEdnluUi1RSFhjNmNadlF5N1FMRGc3aWlUUjhEdw?oc=5) — *Google News / JournalArta*
- `11/10 10:01` [Oil prices fall after President Trump’s statement on Iran — Channel NewsAsia](https://news.google.com/rss/articles/CBMitgFBVV95cUxNTEx2RmNpQVJOd0VTRVFqei1adkFEY1BPQzU2czNVbGJFWW9zNWJ3MGpnTF9IV01pQnNSU2REN05wSklwVEZIWkFkMENfZ3pLZjZSdU9adDdQdkZBb2lhdFdhZ05GdlZEVXhwM3p6aFdCcnRBdG9tYjdGS2dJWXlBblhIclpBSEpCQ1RhOExBLXlZYkJuRlRDbDUwaVFackdqQkpZcWY3YllOVWhxMFlkSDkxTi16UQ?oc=5) — *Google News / UA.NEWS*

### 中文（鉅亨網美股）
- `11/10 11:30` [輝達中國問題早被盯上！「諜報系」神秘空頭半年前就出手 40頁報告掀關注](https://news.cnyes.com/news/id/6655009) — *鉅亨網*：Culper Research 5月警告輝達有中國問題，6個月後40頁報告才受關注。&#10;報告稱黃仁勳與Megaspeed高層黃樂關係「密切得令人不安」，並引述媒體與自家研究。&#10;Culper揭露放空後輝達跌約20%，董事會已調查黃仁勳婚外情與相關國安疑慮。
- `11/10 11:20` [選後首波清算！共和黨人提案彈劾赫格塞斯 川普表態力挺 專家：雖具政治象徵意義 但地緣風險溢價恐延長](https://news.cnyes.com/news/id/6655332) — *鉅亨網*：眾院共和黨議員Massie提案彈劾國防部長赫格塞斯，最快本週表決，焦點是他在對伊朗開戰中的主導角色。&#10;兩黨反對聲浪升溫，川普稱若赫格塞斯遭彈劾將力挺到底，赫格塞斯則預告周二開記者會回應。&#10;分析指彈劾短期難罷免但具政治象徵，恐延長地緣風險溢價，能源與防務股後市受關注。
- `11/10 11:20` [黃仁勳出軌風波牽出資本網絡！女主角黃樂背後投資版圖曝光](https://news.cnyes.com/news/id/6654940) — *鉅亨網*：黃仁勳週日發道歉信承認婚姻不忠；媒體並將黃樂與婚變及輝達合作夥伴Megaspeed連結。&#10;黃樂曾任和熙投資高層，並與上海韻禧投資、第七大道及Megaspeed有股權與董事關係。&#10;和熙投資創辦人康偉亦涉斐訊、世紀華通算力布局，且曾因違規持股與資訊揭露問題遭中國證監會處罰。
- `11/10 11:10` [黃仁勳外遇疑雲延燒 川普喊「他會沒事」 兩人交情遭檢視](https://news.cnyes.com/news/id/6655274) — *鉅亨網*：輝達董事會調查黃仁勳是否遭外國情報機構鎖定，川普也被檢視與其往來&#10;紐時指黃仁勳與中國女商人關係密切；報導後輝達股價跌至7個月低點&#10;黃仁勳近年多次與川普互動，輝達並獲准向中國出口H200等政策利多
- `11/10 11:10` [川普自曝警告某AI大咖「做壞事準備坐牢」 多條線索指向Anthropic執行長](https://news.cnyes.com/news/id/6655271) — *鉅亨網*：川普稱政府因AI不當行為威脅將一名大型AI公司高層送進監獄，但未透露身分&#10;多段發言與6月Anthropic爭議相符，線索指向執行長Dario Amodei&#10;報導指白宮與商務部曾施壓Anthropic下架模型，但目前無司法部逮捕證據
- `11/10 11:10` [頂級富豪婚姻 回顧股神巴菲特「開放式婚姻」](https://news.cnyes.com/news/id/6654550) — *鉅亨網*：巴菲特與蘇珊1952年結婚，1977年蘇珊搬離奧馬哈後，門克斯開始照料其生活，三人長期維持彼此尊重的相處模式。&#10;蘇珊2003年罹口腔癌、2004年中風離世；巴菲特兩人婚姻52年、約27年分居不同城市。&#10;2006年巴菲特76歲生日當天與門克斯結婚；他稱蘇珊讓他成功，門克斯讓成功延續。
- `11/10 11:00` [半導體股暴跌後反彈！大摩點名輝達、SanDisk、美光逢低布局 空方警告還沒跌完](https://news.cnyes.com/news/id/6655279) — *鉅亨網*：半導體股9日強彈，SOXX漲約7%，但華爾街對是否適合進場仍分歧，市場憂AI基建投資與晶片需求放緩&#10;Cantor與大摩看好輝達、SanDisk、美光，稱估值偏低且回購題材可支撐股價，輝達9日仍跌約5%&#10;BTIG指費半跌破200日線屬重要破位，Rob Arnott稱AI變現未明，若未守穩指數恐再修正
- `11/10 10:50` [AMD搶下Google AI晶片關鍵角色！第五代EPYC進駐TPU系統 輝達競爭升溫](https://news.cnyes.com/news/id/6655275) — *鉅亨網*：AMD第五代EPYC獲Google最新TPU 8tx採用，供應主機運算基礎，協助機密運算與資料保護。&#10;TPU 8tx鎖定政府、金融等受監管產業，結合硬體加密；Google稱低延遲最高約提升2倍。&#10;AMD股價9日漲13.3%，Google今年已推4款第八代TPU，TPU客戶近100家。
